# Ours: Bidirectional Antibody-Antigen Language Model
## Comprehensive Analysis on SAaIntDB — Journal Publication Figures

This notebook reproduces all analyses, ablations, and mutagenesis experiments for the Ours paper.

In [ ]:
import os, sys, pickle, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.stats import pearsonr, spearmanr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
warnings.filterwarnings('ignore')

HERE = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, HERE)
from mutual_strong import MutualTriStreamStrong

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Working dir: {HERE}')

In [ ]:
# ── Publication figure style ───────────────────────────────────────────────
NATURE_RC = {
    'font.family':        'Arial',
    'font.size':           7,
    'axes.titlesize':      8,
    'axes.labelsize':      7,
    'xtick.labelsize':     6,
    'ytick.labelsize':     6,
    'legend.fontsize':     6,
    'legend.frameon':      False,
    'axes.linewidth':      0.8,
    'xtick.major.width':   0.8,
    'ytick.major.width':   0.8,
    'xtick.major.size':    3,
    'ytick.major.size':    3,
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'figure.dpi':          400,
    'savefig.dpi':         400,
    'savefig.bbox':        'tight',
    'pdf.fonttype':         42,
    'ps.fonttype':          42,
}
plt.rcParams.update(NATURE_RC)

# Consistent color palette
PALETTE = {
    'BALM_random': '#2ecc71',
    'BALM_cold':   '#27ae60',
    'ESM2':        '#2ecc71',
    'ProtBERT':    '#9b59b6',
    'AntiBERTy':   '#f39c12',
    'ProGen2':     '#3498db',
    'AntiBERTy_ESM2': '#e74c3c',
    'scatter':     '#34495e',
    'highlight':   '#e74c3c',
}

FIG_OUT = os.path.join(HERE, 'figures_publication')
os.makedirs(FIG_OUT, exist_ok=True)

def save_fig(fig, name, formats=('pdf', 'png')):
    for fmt in formats:
        fig.savefig(os.path.join(FIG_OUT, f'{name}.{fmt}'), bbox_inches='tight')
    print(f'Saved: {name}')

def corr_label(x, y):
    mask = ~(np.isnan(x) | np.isnan(y))
    r, _ = pearsonr(x[mask], y[mask])
    rho, _ = spearmanr(x[mask], y[mask])
    return f'r = {r:.3f}\n\u03c1 = {rho:.3f}\nn = {mask.sum()}'

print('Style configured. Figures will be saved to:', FIG_OUT)

---
## 1. Dataset Overview — SAaIntDB

In [ ]:
df_saaint = pd.read_csv(os.path.join(HERE, 'datasets/saaintdb_with_antigen_names.csv'))
print(f'SAaIntDB: {len(df_saaint)} complexes | pK$_d$ [{df_saaint.pKD.min():.2f}, {df_saaint.pKD.max():.2f}]')
print(f'Unique PDB IDs: {df_saaint.PDB_ID.nunique()}')
print(f'Unique antigens: {df_saaint.Antigen_Name.nunique()}')
print(f'Nanobodies (no light chain): {(~df_saaint.L_seq.apply(lambda x: isinstance(x, str) and len(str(x)) > 5)).sum()}')
print(f'Affinity methods: {df_saaint.Affinity_method.value_counts().to_dict()}')

In [ ]:
# Figure 1: Dataset overview
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.2))

# Panel A: pK$_d$ distribution
ax = axes[0]
ax.hist(df_saaint['pKD'], bins=40, color=PALETTE['BALM_random'], edgecolor='white', linewidth=0.3, alpha=0.85)
ax.axvline(df_saaint['pKD'].median(), color='#e74c3c', lw=1.2, linestyle='--', label=f'Median {df_saaint["pKD"].median():.1f}')
ax.set_xlabel(r'pK$_d$ (−log$_{10}$[K$_D$])')
ax.set_ylabel('Count')
ax.set_title('A   pK$_d$ distribution')
ax.legend()

# Panel B: Affinity method
ax = axes[1]
meth = df_saaint['Affinity_method'].value_counts().head(6)
bars = ax.barh(range(len(meth)), meth.values, color=PALETTE['BALM_random'], alpha=0.85, height=0.65)
ax.set_yticks(range(len(meth)))
ax.set_yticklabels(meth.index, fontsize=5.5)
ax.set_xlabel('Number of complexes')
ax.set_title('B   Measurement methods')
for bar, val in zip(bars, meth.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=5.5)

# Panel C: Top antigens
ax = axes[2]
top_ag = df_saaint['Antigen_Name'].value_counts().head(10)
ax.barh(range(len(top_ag)), top_ag.values, color=PALETTE['ProGen2'], alpha=0.85, height=0.65)
ax.set_yticks(range(len(top_ag)))
ax.set_yticklabels([a[:22] for a in top_ag.index], fontsize=5.5)
ax.set_xlabel('Number of complexes')
ax.set_title('C   Top antigen targets')

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig1_dataset_overview')
plt.show()

---
## 2. Main Model Performance — 10-fold CV (Random & Cold splits)

In [ ]:
# Load predictions
df_preds = pd.read_csv(os.path.join(HERE, 'results_mutual_strong_saaintdb/all_folds_predictions.csv'))
print(f'Predictions: {len(df_preds)} rows | cols: {df_preds.columns.tolist()}')

# Summary stats
cv_summary = pd.read_csv(os.path.join(HERE, 'results_mutual_strong_saaintdb/cv_summary.csv'))
print('\nCV Summary:')
print(cv_summary.to_string(index=False))

# Bootstrap CI
bootstrap = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/bootstrap_ci_permutation_tests.csv'))
print('\nBootstrap CI:')
print(bootstrap[bootstrap.model == 'BALM_random'][['metric','point','ci95_low','ci95_high','permutation_pvalue']].to_string(index=False))

In [ ]:
# Figure 2: Main CV performance
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4))

true_pkd = df_preds['binding_affinity'].values
pred_pkd = df_preds['predicted_affinity'].values
r_main,   _ = pearsonr(true_pkd, pred_pkd)
rho_main, _ = spearmanr(true_pkd, pred_pkd)
rmse_main   = np.sqrt(np.mean((true_pkd - pred_pkd)**2))

# Panel A: Scatter true vs predicted
ax = axes[0]
ax.scatter(true_pkd, pred_pkd, c=PALETTE['scatter'], alpha=0.35, s=4, linewidths=0, rasterized=True)
lims = [min(true_pkd.min(), pred_pkd.min()) - 0.3, max(true_pkd.max(), pred_pkd.max()) + 0.3]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.6)
m, b = np.polyfit(true_pkd, pred_pkd, 1)
xs = np.linspace(*lims, 200)
ax.plot(xs, m*xs+b, color=PALETTE['highlight'], lw=1.2)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(r'Measured pK$_d$')
ax.set_ylabel(r'Predicted pK$_d$')
ax.set_title('A   Ours (Ours (Ours-AbAg)) predictions (random CV)')
ax.text(0.05, 0.93, f'r = {r_main:.3f}\n\u03c1 = {rho_main:.3f}\nRMSE = {rmse_main:.3f}',
        transform=ax.transAxes, fontsize=6, va='top')

# Panel B: Error distribution
ax = axes[1]
errors = true_pkd - pred_pkd
ax.hist(errors, bins=50, color=PALETTE['BALM_random'], edgecolor='white', linewidth=0.3, alpha=0.85)
ax.axvline(0, color='k', lw=0.8, linestyle='--')
ax.axvline(errors.mean(), color=PALETTE['highlight'], lw=1.2, label=f'Mean {errors.mean():.3f}')
ax.set_xlabel('Residual (true − predicted) pK$_d$')
ax.set_ylabel('Count')
ax.set_title('B   Residual distribution')
ax.text(0.05, 0.93, f'MAE = {np.abs(errors).mean():.3f}\nStd = {errors.std():.3f}',
        transform=ax.transAxes, fontsize=6, va='top')

# Panel C: Bootstrap CI bar chart for key metrics
ax = axes[2]
b_rand = bootstrap[bootstrap.model == 'BALM_random']
b_cold = bootstrap[bootstrap.model == 'BALM_cold']
metrics_show = ['pearson', 'spearman', 'rmse']
labels_show  = ['Pearson r', 'Spearman \u03c1', r'RMSE (pK$_d$)']
x = np.arange(len(metrics_show))
w = 0.35
for split_df, offset, color, label in [
    (b_rand, -w/2, PALETTE['BALM_random'], 'Random split'),
    (b_cold,  w/2, PALETTE['BALM_cold'],   'Cold split')
]:
    vals = [float(split_df[split_df.metric == m]['point'].values[0]) for m in metrics_show]
    lows = [float(split_df[split_df.metric == m]['ci95_low'].values[0]) for m in metrics_show]
    highs= [float(split_df[split_df.metric == m]['ci95_high'].values[0]) for m in metrics_show]
    yerr = np.array([[v - l, h - v] for v, l, h in zip(vals, lows, highs)]).T
    bars = ax.bar(x + offset, vals, w, color=color, alpha=0.85, label=label,
                  yerr=yerr, error_kw=dict(lw=1, capsize=2, capthick=0.8))
ax.set_xticks(x); ax.set_xticklabels(labels_show)
ax.set_ylabel('Score')
ax.set_title('C   Random vs Cold split (95% CI)')
ax.legend()

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig2_main_cv_performance')
plt.show()

---
## 3. PLM Comparison — ESM-2 vs AntiBERTy vs ProtBERT vs ProGen2

In [ ]:
plm_df = pd.read_csv(os.path.join(HERE, 'plm_comparison_results_mutual_saaintdb/cv_all_plms_saaintdb.csv'))
ablation_df = pd.read_csv(os.path.join(HERE, 'results_ablation_antibody_plm/cv_summary.csv'))
print(plm_df[['PLM','CV_Pearson','CV_Pearson_std','CV_Spearman','CV_RMSE']].to_string(index=False))
print('\nNew ablation (AntiBERTy H/L + ESM-2 Ag):')
print(f'  Pearson r = {ablation_df.pearson.mean():.4f} ± {ablation_df.pearson.std():.4f}')
print(f'  Spearman  = {ablation_df.spearman.mean():.4f}')
print(f'  RMSE      = {ablation_df.rmse.mean():.4f}')

In [ ]:
# Figure 3: PLM comparison (including new AntiBERTy H/L + ESM-2 Ag ablation)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6))

# Add new ablation row
plm_ext = plm_df.copy()
ablation_row = pd.DataFrame([{
    'PLM': 'AntiBERTy-H/L\n+ESM-2-Ag',
    'Color': '#e74c3c',
    'CV_Pearson': ablation_df.pearson.mean(),
    'CV_Pearson_std': ablation_df.pearson.std(),
    'CV_Spearman': ablation_df.spearman.mean(),
    'CV_Spearman_std': ablation_df.spearman.std(),
    'CV_RMSE': ablation_df.rmse.mean(),
    'CV_RMSE_std': ablation_df.rmse.std(),
    'CV_R2': np.nan, 'CV_R2_std': np.nan,
}])
plm_ext = pd.concat([plm_ext, ablation_row], ignore_index=True)

n = len(plm_ext)
x = np.arange(n)
colors = plm_ext['Color'].tolist()
names  = plm_ext['PLM'].tolist()

for ax, col_mean, col_std, ylabel, title, letter in [
    (axes[0], 'CV_Pearson',  'CV_Pearson_std',  'Pearson r',   'A   Pearson r', 'A'),
    (axes[1], 'CV_Spearman', 'CV_Spearman_std', 'Spearman \u03c1', 'B   Spearman \u03c1', 'B'),
    (axes[2], 'CV_RMSE',     'CV_RMSE_std',     r'RMSE (pK$_d$)',   'C   RMSE', 'C'),
]:
    vals = plm_ext[col_mean].values
    errs = plm_ext[col_std].values
    bars = ax.bar(x, vals, color=colors, alpha=0.85, width=0.65,
                  yerr=errs, error_kw=dict(lw=1, capsize=3, capthick=0.8))
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=5.5, rotation=20, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    # Significance annotations vs ESM-2
    esm2_val = float(plm_ext[plm_ext.PLM == 'ESM-2'][col_mean].values[0])
    for i, (val, nm) in enumerate(zip(vals, names)):
        if nm not in ('ESM-2',):
            ax.text(i, val + errs[i] + 0.01, f'{val:.3f}', ha='center', fontsize=5.5)
    ax.text(0, vals[0] + errs[0] + 0.01, f'{vals[0]:.3f}', ha='center', fontsize=5.5)

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig3_plm_comparison')
plt.show()

---
## 4. Stream Intervention Analysis

In [ ]:
stream_df = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/stream_intervention_metrics.csv'))
stream_eb = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/stream_intervention_errorbars.csv'))
print(stream_eb.columns.tolist())
print(stream_eb.head(10).to_string(index=False))

In [ ]:
# Figure 4: Stream intervention (ablation of heavy/light/antigen streams)
conditions_order = ['full', 'zero_light', 'zero_heavy', 'zero_antigen', 'mean_antigen', 'shuffled_antigen']
conditions_label = ['Full model', 'Zero light', 'Zero heavy', 'Zero antigen', 'Mean antigen', 'Shuffled antigen']
cond_colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#95a5a6']

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

for ax, split, title_letter in [(axes[0], 'Random', 'A'), (axes[1], 'Cold', 'B')]:
    sub = stream_df[stream_df['split'] == split]
    # Average over folds per condition
    grp = sub.groupby('condition').agg({'pearson': ['mean', 'std'], 'spearman': ['mean', 'std']}).reset_index()
    grp.columns = ['condition', 'pearson_mean', 'pearson_std', 'spearman_mean', 'spearman_std']
    
    x = np.arange(len(conditions_order))
    for offset, metric_m, metric_s, label in [
        (-0.2, 'pearson_mean',  'pearson_std',  'Pearson r'),
        ( 0.2, 'spearman_mean', 'spearman_std', 'Spearman \u03c1'),
    ]:
        vals, errs = [], []
        for cond in conditions_order:
            row = grp[grp.condition == cond]
            vals.append(float(row[metric_m]) if len(row) else 0)
            errs.append(float(row[metric_s]) if len(row) else 0)
        ax.bar(x + offset, vals, 0.38, yerr=errs, label=label,
               color=['#2ecc71' if offset < 0 else '#3498db'] * len(x),
               alpha=0.8, error_kw=dict(lw=0.8, capsize=2))
    
    ax.set_xticks(x)
    ax.set_xticklabels(conditions_label, rotation=30, ha='right', fontsize=6)
    ax.set_ylabel('Correlation')
    ax.set_title(f'{title_letter}   Stream intervention — {split} split')
    ax.legend()
    ax.axhline(0, color='k', lw=0.5)

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig4_stream_intervention')
plt.show()

---
## 5. Cross-Dataset Generalization — Zero-shot Transfer

In [ ]:
zs_df = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/cross_dataset_zero_shot_metrics.csv'))
print(zs_df[['source','target','n','pearson','spearman','rmse']].to_string(index=False))

In [ ]:
# Figure 5: Cross-dataset heatmap
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))

sources = zs_df['source'].unique()
targets = zs_df['target'].unique()

for ax, metric, title, cmap, vcenter in [
    (axes[0], 'pearson', 'A   Pearson r (zero-shot transfer)', 'RdYlGn', 0),
    (axes[1], 'spearman', 'B   Spearman \u03c1 (zero-shot transfer)', 'RdYlGn', 0),
]:
    matrix = np.full((len(sources), len(targets)), np.nan)
    for i, src in enumerate(sources):
        for j, tgt in enumerate(targets):
            row = zs_df[(zs_df.source == src) & (zs_df.target == tgt)]
            if len(row): matrix[i, j] = float(row[metric].values[0])
    
    vmin, vmax = np.nanmin(matrix), np.nanmax(matrix)
    norm = TwoSlopeNorm(vmin=min(vmin, -0.1), vcenter=vcenter, vmax=max(vmax, 0.5))
    im = ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')
    ax.set_xticks(range(len(targets)))
    ax.set_xticklabels([t.replace('SAaIntDB','SAaIntDB').replace('_',' ') for t in targets],
                       rotation=30, ha='right', fontsize=6)
    ax.set_yticks(range(len(sources)))
    ax.set_yticklabels([s.replace('_random_fold05','\n(random)').replace('_cold_fold05','\n(cold)') for s in sources], fontsize=5.5)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    for i in range(len(sources)):
        for j in range(len(targets)):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center',
                        fontsize=5.5, color='black' if abs(matrix[i,j]) < 0.6 else 'white')

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig5_cross_dataset_transfer')
plt.show()

---
## 6. Few-Shot Learning Curves

In [ ]:
fs_df = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/few_shot_all_available.csv'))
print(fs_df['dataset'].unique())
print(fs_df[fs_df.dataset == 'aayl51'][['mode','fraction','pearson','spearman']].head(10))

In [ ]:
# Figure 6: Few-shot learning curves
datasets = [d for d in fs_df['dataset'].unique() if d not in ('',)]
ds_colors = {'1mlc': '#2ecc71', '4fqi': '#3498db', 'aayl51': '#e74c3c',
             'trastuzumab': '#f39c12', 'warszawski': '#9b59b6', 'koenig': '#e67e22'}

ncols = 3
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(7.2, 4.4), sharey=False)
axes_flat = axes.flatten()
letters = 'ABCDEF'

for idx, ds in enumerate(datasets[:6]):
    ax = axes_flat[idx]
    sub = fs_df[fs_df['dataset'] == ds]
    
    # Zero-shot point
    zshot = sub[sub['mode'] == 'zero_shot']
    if len(zshot):
        zr = zshot['pearson'].mean()
        ax.axhline(zr, color='gray', lw=1, linestyle=':', label=f'Zero-shot ({zr:.3f})')
    
    # Few-shot curves
    fshot = sub[sub['mode'] == 'few_shot']
    if len(fshot) > 0:
        fracs = sorted(fshot['fraction'].dropna().unique())
        grp = fshot.groupby('fraction')['pearson'].agg(['mean','std']).reset_index()
        ax.plot(grp['fraction'], grp['mean'], 'o-', color=ds_colors.get(ds, '#2c3e50'),
                lw=1.5, ms=3, label='Few-shot')
        ax.fill_between(grp['fraction'],
                         grp['mean'] - grp['std'],
                         grp['mean'] + grp['std'],
                         alpha=0.2, color=ds_colors.get(ds, '#2c3e50'))
    
    ax.set_xlabel('Training fraction')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'{letters[idx]}   {ds.upper()}')
    ax.legend(fontsize=5.5)
    ax.axhline(0, color='k', lw=0.5, linestyle='--', alpha=0.4)

plt.tight_layout()
save_fig(fig, 'fig6_few_shot_learning')
plt.show()

---
## 7. Per-Antigen Intra-Target Ranking

In [ ]:
intra_df = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/per_antigen_intra_target_metrics.csv'))
print(f'Intra-target metrics: {len(intra_df)} antigen groups')
# Filter to groups with at least 3 complexes
intra_df = intra_df[intra_df['n'] >= 3].copy()
print(f'Groups with n>=3: {len(intra_df)}')
print(intra_df[['antigen','n','pearson','spearman']].sort_values('spearman', ascending=False).head(10).to_string(index=False))

In [ ]:
# Figure 7: Per-antigen intra-target analysis
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))

rand_intra = intra_df[intra_df['split'] == 'Random'] if 'split' in intra_df.columns else intra_df

# Panel A: Distribution of per-antigen Spearman rho
ax = axes[0]
ax.hist(rand_intra['spearman'].dropna(), bins=20, color=PALETTE['BALM_random'], edgecolor='white', linewidth=0.3)
ax.axvline(rand_intra['spearman'].median(), color=PALETTE['highlight'], lw=1.2, linestyle='--',
           label=f'Median {rand_intra["spearman"].median():.3f}')
ax.set_xlabel('Intra-target Spearman \u03c1')
ax.set_ylabel('Count')
ax.set_title('A   Per-antigen ranking performance')
ax.legend()

# Panel B: Pearson vs n complexes per antigen
ax = axes[1]
ax.scatter(rand_intra['n'], rand_intra['pearson'], c=PALETTE['scatter'], alpha=0.5, s=12)
ax.axhline(0, color='k', lw=0.5, linestyle='--')
ax.set_xlabel('Number of complexes per antigen')
ax.set_ylabel('Pearson r')
ax.set_title('B   Sample size vs performance')

# Panel C: Top 15 antigens by Spearman rho
ax = axes[2]
top15 = rand_intra.dropna(subset=['spearman']).nlargest(15, 'spearman')
colors_bar = [PALETTE['BALM_random'] if r > 0 else PALETTE['highlight'] for r in top15['spearman']]
ax.barh(range(len(top15)), top15['spearman'], color=colors_bar, alpha=0.85, height=0.7)
ax.set_yticks(range(len(top15)))
pdb_labels = top15['pdb'].values if 'pdb' in top15.columns else top15['antigen'].values
ax.set_yticklabels([str(p)[:12] for p in pdb_labels], fontsize=5.5)
ax.set_xlabel('Spearman \u03c1')
ax.set_title('C   Top-15 antigens')
ax.axvline(0, color='k', lw=0.5)

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig7_per_antigen_intra_target')
plt.show()

---
## 8. New Ablation: AntiBERTy H/L + ESM-2 Ag (fold-level detail)

In [ ]:
ablation_folds = pd.read_csv(os.path.join(HERE, 'results_ablation_antibody_plm/cv_summary.csv'))
print(ablation_folds)

# Build a fold-level comparison table
fold_errorbars = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/fold_level_errorbars_all_models.csv'))
baseline_folds = fold_errorbars[fold_errorbars['model'] == 'Ours (random split)'][['metric','mean','std']]
print('\nBaseline per-metric:')
print(baseline_folds.to_string(index=False))

In [ ]:
# Figure 8: Detailed ablation comparison
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))

# Get baseline means/stds from fold_errorbars
def get_metric(df, model, metric):
    row = df[(df.model == model) & (df.metric == metric)]
    if len(row): return float(row['mean'].values[0]), float(row['std'].values[0])
    return np.nan, np.nan

bl_models = fold_errorbars['model'].unique()
metric_cols = [('pearson', 'Pearson r', 'A'), ('spearman', 'Spearman \u03c1', 'B'), ('rmse', r'RMSE (pK$_d$)', 'C')]

for ax, (metric, ylabel, letter) in zip(axes, metric_cols):
    # All baseline models from fold_errorbars
    base_rows = []
    for m in bl_models:
        mean, std = get_metric(fold_errorbars, m, metric)
        base_rows.append({'model': m, 'mean': mean, 'std': std})
    base_df = pd.DataFrame(base_rows).dropna()
    
    # Add new ablation
    abl_mean = ablation_folds[metric].mean() if metric in ablation_folds.columns else np.nan
    abl_std  = ablation_folds[metric].std()  if metric in ablation_folds.columns else np.nan
    new_row = pd.DataFrame([{'model': 'AntiBERTy H/L\n+ESM-2 Ag', 'mean': abl_mean, 'std': abl_std}])
    combined = pd.concat([base_df, new_row], ignore_index=True)
    
    combined = combined.sort_values('mean', ascending=(metric=='rmse'))
    colors_bar = [PALETTE['highlight'] if 'AntiBERTy' in m else '#7f8c8d' if 'cold' in m.lower() else PALETTE['BALM_random']
                  for m in combined['model']]
    ax.barh(range(len(combined)), combined['mean'], color=colors_bar, alpha=0.85, height=0.7,
            xerr=combined['std'], error_kw=dict(lw=0.8, capsize=2))
    ax.set_yticks(range(len(combined)))
    ax.set_yticklabels(combined['model'], fontsize=5.5)
    ax.set_xlabel(ylabel)
    ax.set_title(f'{letter}   {ylabel}')
    ax.axvline(0, color='k', lw=0.5)

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig8_ablation_model_comparison')
plt.show()

---
## 9. Mutagenesis Analysis — Koenig 2017 G6 Antibody Dataset

In [ ]:
# Load Koenig dataset
df_koenig = pd.read_csv(os.path.join(HERE, 'datasets/koenig2017mutational_kd_g6.csv'))
df_koenig.columns = ['heavy', 'light', 'delta_log_kd', 'KD_nM', 'KD_M', 'log_kd', 'pKD', 'antigen']
df_koenig['pKD'] = df_koenig['pKD'].astype(float)
df_koenig['KD_nM'] = df_koenig['KD_nM'].astype(float)

print(f'Koenig dataset: {len(df_koenig)} variants')
print(f'pK$_d$ range: {df_koenig.pKD.min():.2f} – {df_koenig.pKD.max():.2f}')
print(f'Wild-type heavy: {df_koenig.heavy.iloc[0][:60]}...')

# Identify mutation positions: which position differs from WT
wt_heavy = df_koenig['heavy'].iloc[0]
def find_mutation(h, wt):
    diffs = [(i, wt[i], h[i]) for i in range(min(len(h), len(wt))) if h[i] != wt[i]]
    return diffs[0] if diffs else None

df_koenig['mutation'] = df_koenig['heavy'].apply(lambda h: find_mutation(h, wt_heavy))
df_koenig['mut_pos'] = df_koenig['mutation'].apply(lambda m: m[0] if m else -1)
df_koenig['wt_aa']   = df_koenig['mutation'].apply(lambda m: m[1] if m else '=')
df_koenig['mut_aa']  = df_koenig['mutation'].apply(lambda m: m[2] if m else '=')

positions_scanned = df_koenig['mut_pos'].value_counts()
print(f'\nPositions scanned: {len(positions_scanned)}')
print(f'Variants per position (top 5): {positions_scanned.head(5).to_dict()}')

In [ ]:
# Run BALM predictions on Koenig dataset
# Use the SAINTdb-trained model ensemble (zero-shot transfer)
import glob as _glob
from transformers import EsmTokenizer, EsmModel

def compute_esm2_embeddings_batch(sequences, device, batch_size=8,
                                   model_name='facebook/esm2_t33_650M_UR50D'):
    """Compute ESM-2 mean-pool embeddings for a list of sequences."""
    tokenizer = EsmTokenizer.from_pretrained(model_name)
    esm_model = EsmModel.from_pretrained(model_name).to(device)
    esm_model.eval()
    embeddings = {}
    unique_seqs = list(dict.fromkeys(sequences))
    for i in range(0, len(unique_seqs), batch_size):
        batch = unique_seqs[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = esm_model(**inputs)
        token_embs = out.last_hidden_state
        attn_mask  = inputs['attention_mask']
        for j, seq in enumerate(batch):
            seq_len = int(attn_mask[j].sum().item()) - 2
            emb = token_embs[j, 1:seq_len+1, :].mean(0).cpu().numpy()
            embeddings[seq] = emb
        if (i // batch_size) % 5 == 0:
            print(f'  Embedded {min(i+batch_size, len(unique_seqs))}/{len(unique_seqs)}', end='\r')
    del esm_model; torch.cuda.empty_cache()
    return embeddings

# Only compute for a representative subset to keep runtime manageable
# Select: wildtype + all variants at the top-5 most-scanned positions
top_pos = positions_scanned[positions_scanned >= 15].index.tolist()
df_sub = df_koenig[(df_koenig['mut_pos'].isin(top_pos)) | (df_koenig['mut_pos'] == -1)].copy().reset_index(drop=True)
print(f'Subset for BALM inference: {len(df_sub)} variants (positions: {top_pos})')

print('Computing ESM-2 embeddings...')
all_seqs = list(df_sub['heavy'].unique()) + list(df_sub['light'].unique()) + list(df_sub['antigen'].unique())
emb_dict = compute_esm2_embeddings_batch(all_seqs, DEVICE)
print(f'\nEmbedded {len(emb_dict)} unique sequences')

In [ ]:
# BALM inference: ensemble of 10 SAINTdb folds
def load_balm_model(ckpt_path, device):
    ck  = torch.load(ckpt_path, map_location=device)
    cfg = ck.get('config', {})
    m   = MutualTriStreamStrong(
        esm_dim=1280,
        projected_size=cfg.get('projected_size', 256),
        num_heads=cfg.get('num_heads', 8),
        dropout=cfg.get('dropout', 0.1),
        n_layers=cfg.get('n_layers', 2),
        device=device,
    ).to(device)
    m.load_state_dict(ck['model_state_dict'])
    return m, ck['pkd_bounds']

@torch.no_grad()
def predict_single(model, h_emb, l_emb, a_emb, bounds, device):
    lo, hi = bounds
    he = torch.tensor(h_emb, dtype=torch.float32).unsqueeze(0).to(device)
    le = torch.tensor(l_emb, dtype=torch.float32).unsqueeze(0).to(device)
    ae = torch.tensor(a_emb, dtype=torch.float32).unsqueeze(0).to(device)
    cos = model(le, he, ae)['cosine_similarity'].item()
    return (cos + 1.0) / 2.0 * (hi - lo) + lo

fold_ckpts = sorted(_glob.glob(os.path.join(HERE, 'results_mutual_strong_saaintdb/fold_*/model.pt')))
print(f'Found {len(fold_ckpts)} SAINTdb folds')

all_fold_preds = []
for ck_path in fold_ckpts:
    model, bounds = load_balm_model(ck_path, DEVICE)
    model.eval()
    fold_preds = []
    for _, row in df_sub.iterrows():
        h_emb = emb_dict.get(row['heavy'])
        l_emb = emb_dict.get(row['light'])
        a_emb = emb_dict.get(row['antigen'])
        if h_emb is None or l_emb is None or a_emb is None:
            fold_preds.append(np.nan)
        else:
            fold_preds.append(predict_single(model, h_emb, l_emb, a_emb, bounds, DEVICE))
    all_fold_preds.append(fold_preds)
    fold_num = os.path.basename(os.path.dirname(ck_path))
    r_fold, _ = pearsonr(df_sub['pKD'].values,
                          np.array([p for p in fold_preds if not np.isnan(p)])[:len(df_sub)])
    print(f'  {fold_num}: r={r_fold:.4f}')

df_sub['balm_pred_ensemble'] = np.nanmean(all_fold_preds, axis=0)
df_sub['balm_pred_std']      = np.nanstd(all_fold_preds, axis=0)

mask = df_sub['balm_pred_ensemble'].notna()
r_ens, _   = pearsonr(df_sub.loc[mask, 'pKD'],  df_sub.loc[mask, 'balm_pred_ensemble'])
rho_ens, _ = spearmanr(df_sub.loc[mask, 'pKD'], df_sub.loc[mask, 'balm_pred_ensemble'])
print(f'\nEnsemble (zero-shot): Pearson r={r_ens:.4f}  Spearman rho={rho_ens:.4f}')

In [ ]:
# Figure 9: Mutagenesis analysis
fig = plt.figure(figsize=(7.2, 5.5))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel A: pK$_d$ distribution of Koenig variants
ax = fig.add_subplot(gs[0, 0])
ax.hist(df_koenig['pKD'], bins=35, color='#3498db', edgecolor='white', lw=0.3, alpha=0.85)
ax.axvline(df_koenig['pKD'].iloc[0], color='k', lw=1.5, linestyle='--',
           label=f'WT pKD={df_koenig["pKD"].iloc[0]:.2f}')
ax.set_xlabel(r'pK$_d$'); ax.set_ylabel('Count')
ax.set_title('A   Koenig G6 variants\n(4275 CDR mutations)')
ax.legend()

# Panel B: Measured ΔpKD at each mutation position
ax = fig.add_subplot(gs[0, 1])
wt_pkd = df_koenig.loc[df_koenig['mut_pos'] == -1, 'pKD'].values
wt_pkd_val = wt_pkd[0] if len(wt_pkd) > 0 else df_koenig['pKD'].iloc[0]
df_koenig['delta_pkd_measured'] = df_koenig['pKD'] - wt_pkd_val
pos_grp = df_koenig[df_koenig['mut_pos'] >= 0].groupby('mut_pos')['delta_pkd_measured'].agg(['mean','std','count'])
pos_grp = pos_grp[pos_grp['count'] >= 5].sort_index()
colors_pos = ['#e74c3c' if m > 0 else '#3498db' for m in pos_grp['mean']]
ax.bar(range(len(pos_grp)), pos_grp['mean'], color=colors_pos, alpha=0.8,
       yerr=pos_grp['std'], error_kw=dict(lw=0.5, capsize=1.5))
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Position index'); ax.set_ylabel('Mean ΔpKD (mut − WT)')
ax.set_title('B   Mean ΔpKD per scanned position\n(red=improvement, blue=decrease)')
ax.set_xticks(range(len(pos_grp)))
ax.set_xticklabels(pos_grp.index.tolist(), fontsize=4.5, rotation=90)

# Panel C: BALM ensemble predictions vs measured pK$_d$
ax = fig.add_subplot(gs[0, 2])
mask = df_sub['balm_pred_ensemble'].notna()
true = df_sub.loc[mask, 'pKD'].values
pred = df_sub.loc[mask, 'balm_pred_ensemble'].values
ax.scatter(true, pred, c='#34495e', alpha=0.5, s=10, linewidths=0, rasterized=True)
lims = [min(true.min(), pred.min()) - 0.2, max(true.max(), pred.max()) + 0.2]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel(r'Measured pK$_d$'); ax.set_ylabel(r'Ours predicted pK$_d$')
ax.set_title('C   Ours zero-shot\non Koenig variants')
ax.text(0.05, 0.93, f'r = {r_ens:.3f}\n\u03c1 = {rho_ens:.3f}\nn={mask.sum()}',
        transform=ax.transAxes, fontsize=6, va='top')

# Panel D: Mutation heatmap for a key position
# Pick the position with most variants and strongest spread
best_pos = pos_grp['std'].idxmax()
pos_variants = df_koenig[df_koenig['mut_pos'] == best_pos][['mut_aa','delta_pkd_measured']].set_index('mut_aa')
ax = fig.add_subplot(gs[1, 0])
AAs = list('ACDEFGHIKLMNPQRSTVWY')
def _get_delta(aa):
    if aa not in pos_variants.index: return 0.0
    v = pos_variants.loc[aa, 'delta_pkd_measured']
    return float(v.values[0]) if hasattr(v, 'values') else float(v)
delta_vals = [_get_delta(aa) for aa in AAs]
wt_aa_pos = df_koenig[df_koenig['mut_pos'] == best_pos]['wt_aa'].iloc[0]
bar_colors = ['#e74c3c' if d > 0 else '#3498db' for d in delta_vals]
ax.bar(range(len(AAs)), delta_vals, color=bar_colors, alpha=0.85, width=0.75)
ax.axhline(0, color='k', lw=0.6)
wt_idx = AAs.index(wt_aa_pos) if wt_aa_pos in AAs else -1
if wt_idx >= 0:
    ax.bar(wt_idx, delta_vals[wt_idx], color='gray', alpha=0.9, width=0.75, label='WT')
ax.set_xticks(range(len(AAs))); ax.set_xticklabels(AAs, fontsize=6)
ax.set_xlabel('Substituted amino acid')
ax.set_ylabel(r'ΔpK$_d$ (mut−WT)')
ax.set_title(f'D   Single-position scan\nPosition {best_pos} (WT={wt_aa_pos})')
ax.legend(fontsize=6)

# Panel E: Predicted ΔpKD vs measured ΔpKD for subset
ax = fig.add_subplot(gs[1, 1])
df_sub['delta_pkd_measured'] = df_sub['pKD'] - wt_pkd_val
df_sub['delta_pkd_pred']     = df_sub['balm_pred_ensemble'] - df_sub.loc[
    df_sub['mut_pos'] == -1, 'balm_pred_ensemble'].mean() if (df_sub['mut_pos'] == -1).any() else 0
mask2 = df_sub['balm_pred_ensemble'].notna() & (df_sub['mut_pos'] >= 0)
if mask2.sum() >= 3:
    dm = df_sub.loc[mask2, 'delta_pkd_measured'].values
    dp = df_sub.loc[mask2, 'delta_pkd_pred'].values
    r_delta, _ = pearsonr(dm, dp)
    rho_delta, _ = spearmanr(dm, dp)
    ax.scatter(dm, dp, c='#8e44ad', alpha=0.5, s=12, linewidths=0, rasterized=True)
    ax.axhline(0, color='k', lw=0.5, linestyle='--', alpha=0.5)
    ax.axvline(0, color='k', lw=0.5, linestyle='--', alpha=0.5)
    ax.set_xlabel('Measured ΔpKD'); ax.set_ylabel('Predicted ΔpKD')
    ax.set_title('E   Predicted vs measured ΔpKD')
    ax.text(0.05, 0.93, f'r = {r_delta:.3f}\n\u03c1 = {rho_delta:.3f}',
            transform=ax.transAxes, fontsize=6, va='top')
else:
    ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)

# Panel F: Per-position ranking (Spearman rho of predicted vs measured within each position)
ax = fig.add_subplot(gs[1, 2])
intra_pos_corrs = []
for pos in top_pos:
    pos_sub = df_sub[df_sub['mut_pos'] == pos].dropna(subset=['balm_pred_ensemble'])
    if len(pos_sub) >= 5:
        rho_p, _ = spearmanr(pos_sub['pKD'], pos_sub['balm_pred_ensemble'])
        intra_pos_corrs.append({'position': pos, 'spearman': rho_p, 'n': len(pos_sub)})
if intra_pos_corrs:
    ipc = pd.DataFrame(intra_pos_corrs)
    ax.barh(range(len(ipc)), ipc['spearman'],
            color=[PALETTE['BALM_random'] if r > 0 else PALETTE['highlight'] for r in ipc['spearman']],
            alpha=0.85, height=0.7)
    ax.set_yticks(range(len(ipc)))
    ax.set_yticklabels([f'Pos {int(p)}' for p in ipc['position']], fontsize=6)
    ax.axvline(0, color='k', lw=0.6)
    ax.set_xlabel('Intra-position Spearman \u03c1')
    ax.set_title('F   Per-position ranking\n(Ours zero-shot)')
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)

save_fig(fig, 'fig9_mutagenesis_analysis')
plt.show()

---
## 10. SAaIntDB Natural Variant Mutation Effects

In [ ]:
# Load SAINTdb mutation effects (computed by gen_figures_saaintdb.py)
saaint_mut = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb_mutation/saaintdb_mutation_effects.csv'))
# Filter to meaningful mutations (non-zero delta)
saaint_mut = saaint_mut[saaint_mut['delta_affinity'].abs() > 0.001].copy()
print(f'SAaIntDB mutations: {len(saaint_mut)}')
print(saaint_mut.head(5).to_string(index=False))

# Load alanine scanning
alanine_df = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb_mutation/alanine_scanning_results.csv'))
print(f'\nAlanine scanning: {len(alanine_df)} positions')
print(alanine_df[alanine_df['delta_affinity'].abs() > 0].head(10).to_string(index=False))

In [ ]:
# Figure 10: SAaIntDB mutation effects
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))

# Panel A: Delta affinity distribution by chain
ax = axes[0]
chain_colors = {'H': '#e74c3c', 'L': '#3498db', 'A': '#2ecc71'}
for chain in saaint_mut['chain'].unique():
    data = saaint_mut[saaint_mut['chain'] == chain]['delta_affinity']
    ax.hist(data, bins=30, alpha=0.7, label=f'Chain {chain} (n={len(data)})',
            color=chain_colors.get(chain, '#95a5a6'), density=True, edgecolor='white', lw=0.2)
ax.axvline(0, color='k', lw=0.8, linestyle='--')
ax.set_xlabel('Predicted ΔpKD')
ax.set_ylabel('Density')
ax.set_title('A   Mutation effect distribution\nby chain (SAaIntDB complexes)')
ax.legend()

# Panel B: Top affinity-improving substitutions across all complexes
ax = axes[1]
top_improve = saaint_mut.nlargest(15, 'delta_affinity')
colors_imp = [chain_colors.get(c, '#95a5a6') for c in top_improve['chain']]
ax.barh(range(len(top_improve)), top_improve['delta_affinity'], color=colors_imp, alpha=0.85, height=0.7)
ax.set_yticks(range(len(top_improve)))
labels_imp = [f"{row['wt_aa']}{row['position']}{row['mut_aa']} ({row['pdb_id']})"
              for _, row in top_improve.iterrows()]
ax.set_yticklabels(labels_imp, fontsize=5)
ax.set_xlabel('Predicted ΔpKD')
ax.set_title('B   Top affinity-improving\nmutations')

# Panel C: Alanine scanning — positions with largest effect
ax = axes[2]
alanine_nonzero = alanine_df[alanine_df['delta_affinity'].abs() > 0].copy()
if len(alanine_nonzero) > 0:
    top_alanine = alanine_nonzero.reindex(alanine_nonzero['delta_affinity'].abs().nlargest(20).index)
    colors_ala = [chain_colors.get(c, '#95a5a6') for c in top_alanine['chain']]
    ax.barh(range(len(top_alanine)), top_alanine['delta_affinity'], color=colors_ala, alpha=0.85, height=0.7)
    ax.set_yticks(range(len(top_alanine)))
    labels_ala = [f"{row['original_aa']}{row['position']}A ({row['chain']})"
                  for _, row in top_alanine.iterrows()]
    ax.set_yticklabels(labels_ala, fontsize=5)
    ax.set_xlabel('Predicted ΔpKD (→Ala)')
    ax.set_title('C   Alanine scanning\n(hotspot residues)')
    ax.axvline(0, color='k', lw=0.5)

# Add legend for chain colors
patches = [mpatches.Patch(color=v, label=f'Chain {k}') for k, v in chain_colors.items()]
axes[2].legend(handles=patches, fontsize=6)

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig10_saaintdb_mutation_effects')
plt.show()

---
## 11. Summary Table — All Results

In [ ]:
# Comprehensive summary table
summary_rows = []

# Main model
summary_rows.append({'Experiment': 'Ours (Ours) (random CV)', 'Pearson r': f'{r_main:.4f}',
                     'Spearman ρ': f'{rho_main:.4f}', r'RMSE (pK$_d$)': f'{rmse_main:.4f}', 'n': 2575})

# Bootstrap results
br = bootstrap[bootstrap.model == 'BALM_random']
for _, row in br.iterrows():
    if row.metric in ('pearson', 'spearman', 'rmse'):
        summary_rows.append({'Experiment': f'Ours (Ours) bootstrap ({row.metric})',
                             'Pearson r': f'{row.point:.4f}',
                             'Spearman ρ': f'95%CI [{row.ci95_low:.4f}, {row.ci95_high:.4f}]',
                             r'RMSE (pK$_d$)': f'p={row.permutation_pvalue:.4f}', 'n': int(row.n)})

# Cold split
bc = bootstrap[bootstrap.model == 'BALM_cold']
pr = bc[bc.metric=='pearson']['point'].values
if len(pr): summary_rows.append({'Experiment': 'Ours (Ours) (cold CV)', 'Pearson r': f'{pr[0]:.4f}',
                                  'Spearman ρ': '', r'RMSE (pK$_d$)': '', 'n': 2575})

# PLM comparison
for _, row in plm_df.iterrows():
    summary_rows.append({'Experiment': f'PLM: {row.PLM}',
                         'Pearson r': f'{row.CV_Pearson:.4f}±{row.CV_Pearson_std:.4f}',
                         'Spearman ρ': f'{row.CV_Spearman:.4f}',
                         r'RMSE (pK$_d$)': f'{row.CV_RMSE:.4f}', 'n': 2575})

# New ablation
summary_rows.append({'Experiment': 'Ablation: AntiBERTy H/L + ESM-2 Ag',
                     'Pearson r': f'{ablation_folds.pearson.mean():.4f}±{ablation_folds.pearson.std():.4f}',
                     'Spearman ρ': f'{ablation_folds.spearman.mean():.4f}',
                     r'RMSE (pK$_d$)': f'{ablation_folds.rmse.mean():.4f}', 'n': 2575})

# Koenig mutagenesis
if 'r_ens' in dir():
    summary_rows.append({'Experiment': 'Koenig 2017 (zero-shot)',
                         'Pearson r': f'{r_ens:.4f}', 'Spearman ρ': f'{rho_ens:.4f}',
                         r'RMSE (pK$_d$)': 'N/A', 'n': mask.sum()})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
# Figure 11: Summary comparison figure (publication-ready)
fig, ax = plt.subplots(figsize=(7.2, 3.5))

model_names = ['ESM-2\n(baseline)','ProtBERT','AntiBERTy','ProGen2','AntiBERTy H/L\n+ESM-2 Ag']
pearson_means = [float(plm_df[plm_df.PLM==p]['CV_Pearson'].values[0]) if p in plm_df.PLM.values
                 else ablation_folds.pearson.mean() for p in ['ESM-2','ProtBERT','AntiBERTy','ProGen2','ABLATION']]
pearson_stds  = [float(plm_df[plm_df.PLM==p]['CV_Pearson_std'].values[0]) if p in plm_df.PLM.values
                 else ablation_folds.pearson.std() for p in ['ESM-2','ProtBERT','AntiBERTy','ProGen2','ABLATION']]

x = np.arange(len(model_names))
bar_colors = [PALETTE['ESM2'], PALETTE['ProtBERT'], PALETTE['AntiBERTy'],
              PALETTE['ProGen2'], PALETTE['AntiBERTy_ESM2']]
bars = ax.bar(x, pearson_means, color=bar_colors, alpha=0.87, width=0.65,
              yerr=pearson_stds, error_kw=dict(lw=1, capsize=4, capthick=1))

# Add significance bracket vs ESM-2
esm2_mean = pearson_means[0]
for i in range(1, len(model_names)):
    delta = pearson_means[i] - esm2_mean
    ax.text(i, pearson_means[i] + pearson_stds[i] + 0.01,
            f'Δ{delta:+.3f}', ha='center', fontsize=5.5,
            color='#e74c3c' if abs(delta) > 0.05 else '#7f8c8d')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=7)
ax.set_ylabel('10-fold CV Pearson r', fontsize=8)
ax.set_title('Ours (Ours (Ours-AbAg)): PLM Ablation Study — SAaIntDB (n=2575 complexes)', fontsize=9, fontweight='bold')
ax.axhline(esm2_mean, color='gray', lw=0.8, linestyle='--', alpha=0.5, label='ESM-2 reference')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=6)

# Add value labels on bars
for bar, val, std in zip(bars, pearson_means, pearson_stds):
    ax.text(bar.get_x() + bar.get_width()/2, val/2, f'{val:.3f}',
            ha='center', va='center', fontsize=6.5, color='white', fontweight='bold')

plt.tight_layout()
save_fig(fig, 'fig11_summary_plm_ablation')
plt.show()

---
## 11b. SAaIntDB — All-CDR vs Mean-Pool Comparison

ANARCI IMGT CDR detection on 1385 unique heavy chains; token-level pooling over CDR-H1+H2+H3 positions.

In [ ]:
# Figure 11b: SAaIntDB All-CDR vs mean-pool
saaintdb_cdr_path = os.path.join(HERE, 'results_saaintdb_allcdr/saaintdb_allcdr_comparison.csv')
if not os.path.isfile(saaintdb_cdr_path):
    print("SAaIntDB All-CDR results not yet available — still running.")
    print("Check: results_saaintdb_allcdr/saaintdb_allcdr_comparison.csv")
else:
    cmp = pd.read_csv(saaintdb_cdr_path)
    print(cmp.to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
    fig.suptitle('SAaIntDB: Mean-pool ESM-2 vs CDR-aware (All-CDR H1+H2+H3)\n'
                 'ANARCI IMGT numbering — 10-fold CV', fontsize=9, fontweight='bold')

    for ax, split, letter, title in [
        (axes[0], 'random', 'A', 'Random split'),
        (axes[1], 'cold',   'B', 'Cold split (no PDB overlap)'),
    ]:
        sub = cmp[cmp['split'] == split]
        methods_s = sub['method'].tolist()
        clrs = ['#95a5a6', '#e74c3c']
        xs   = np.arange(len(methods_s))
        ax.bar(xs, sub['pearson'].values, color=clrs, alpha=0.87, width=0.5)
        ax.set_xticks(xs); ax.set_xticklabels(methods_s, fontsize=6.5, rotation=15, ha='right')
        ax.set_ylabel('Pearson r'); ax.set_title(title, fontsize=8)
        for xi, (_, row) in enumerate(sub.iterrows()):
            ax.text(xi, row['pearson']+0.005, f'{row["pearson"]:.3f}',
                    ha='center', fontsize=7, fontweight='bold')
        ax.set_ylim(0, max(sub['pearson'].values)*1.3)
        ax.text(0.02, 0.97, letter, transform=ax.transAxes,
                fontsize=10, fontweight='bold', va='top')
        # Delta annotation
        if len(sub) == 2:
            delta = float(sub.iloc[1]['pearson']) - float(sub.iloc[0]['pearson'])
            ax.annotate(f'Δ={delta:+.3f}', xy=(0.5, 0.88),
                        xycoords='axes fraction', ha='center', fontsize=7,
                        color='#e74c3c' if delta > 0 else '#3498db',
                        fontweight='bold')

    # Panel C: Per-fold breakdown
    rand_fold = pd.read_csv(os.path.join(HERE, 'results_saaintdb_allcdr/random/cv_summary.csv'))
    base_fold = pd.read_csv(os.path.join(HERE, 'results_mutual_strong_saaintdb/all_folds_predictions.csv'))
    ax = axes[2]
    folds_x = np.arange(1, len(rand_fold)+1)
    ax.plot(folds_x, rand_fold['pearson'], 'o-', color='#e74c3c', lw=1.5, ms=5, label='All-CDR')
    # baseline per-fold from cv_summary
    baseline_cv = pd.read_csv(os.path.join(HERE, 'results_mutual_strong_saaintdb/cv_summary.csv'))
    bl_mean = float(baseline_cv[baseline_cv.metric=='pearson']['mean'].values[0])
    bl_std  = float(baseline_cv[baseline_cv.metric=='pearson']['std'].values[0])
    ax.axhline(bl_mean, color='#95a5a6', lw=1.5, ls='--', label=f'Mean-pool ({bl_mean:.3f})')
    ax.fill_between(folds_x, bl_mean-bl_std, bl_mean+bl_std, alpha=0.15, color='#95a5a6')
    ax.set_xlabel('Fold'); ax.set_ylabel('Pearson r')
    ax.set_title('C   Per-fold (random split)', fontsize=8)
    ax.legend(fontsize=6.5); ax.set_xticks(folds_x); ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.97, 'C', transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

    plt.tight_layout(w_pad=1.5)
    save_fig(fig, 'fig11b_saaintdb_allcdr')
    plt.show()

In [ ]:
print('\nAll publication figures saved to:', FIG_OUT)
import os
figs = sorted(os.listdir(FIG_OUT))
for f in figs:
    print(f'  {f}')

---
## 12. CDR-Aware Embedding Ablation — AAYL51

Comparing full-chain vs CDR-H3 only vs All-CDR (H1+H2+H3) pooling for both in-domain training and zero-shot transfer from SAINTdb.

In [ ]:
# CDR-aware ablation results
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import pearsonr, spearmanr

ADV = os.path.join(HERE, 'advanced_results')
cdr_abl = pd.read_csv(os.path.join(ADV, 'cdr_ablation_results.csv'))
print(cdr_abl.to_string(index=False))


In [ ]:
# Figure 12: CDR-aware embedding ablation — proper panel layout
cdr_abl = pd.read_csv(os.path.join(ADV, 'cdr_ablation_results.csv'))
tl      = pd.read_csv(os.path.join(ADV, 'transfer_learning_curve.csv'))
print("CDR ablation:"); print(cdr_abl[['mode','indomain_r','zeroshot_r']].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))
modes  = cdr_abl['mode'].tolist()
colors = ['#95a5a6', '#f39c12', '#e74c3c']
x      = np.arange(len(modes))
panel_letters = 'ABC'

for ax, metric_col, zs_col, ylabel, title, letter in [
    (axes[0], 'indomain_r',   'zeroshot_r', 'Pearson r',
     'In-domain training (80% AAYL51)', 'A'),
    (axes[1], 'indomain_rho', None,         'Spearman \u03c1',
     'In-domain training (Spearman)', 'B'),
]:
    vals = cdr_abl[metric_col].values
    bars = ax.bar(x, vals, color=colors, alpha=0.87, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(modes, fontsize=6, rotation=18, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=7.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f'{v:.3f}',
                ha='center', fontsize=6.5, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.25)
    ax.axhline(vals[0], color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

# Panel C: zero-shot comparison
ax = axes[2]
zs_vals = cdr_abl['zeroshot_r'].values
bars = ax.bar(x, zs_vals, color=colors, alpha=0.87, width=0.6)
ax.set_xticks(x); ax.set_xticklabels(modes, fontsize=6, rotation=18, ha='right')
ax.set_ylabel('Pearson r'); ax.set_title('Zero-shot (SAINTdb pre-trained)', fontsize=7.5)
for bar, v in zip(bars, zs_vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.3f}', ha='center', fontsize=6.5)
ax.text(0.02, 0.97, 'C', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top')
ax.set_ylim(0, max(max(zs_vals), 0.15)*1.4)

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig12_cdr_aware_ablation')
plt.show()

---
## 13. Transfer Learning: SAINTdb → AAYL51

Starting from SAINTdb pre-trained weights, fine-tune with increasing fractions of AAYL51 training data. Shows the value of domain adaptation.

In [ ]:
tl = pd.read_csv(os.path.join(ADV, 'transfer_learning_curve.csv'))
print(tl[['fraction','n_train','pearson_r','spearman_rho','rmse']].to_string(index=False))


In [ ]:
# Figure 13: Transfer learning curve
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))

ft = tl[tl['fraction'] > 0]
zs = float(tl[tl['fraction'] == 0]['pearson_r'].values[0])
full_r = 0.4228  # in-domain full training (from proper_comparison)

for ax, metric, ylabel, title, letter in [
    (axes[0], 'pearson_r',     'Pearson r',    'A   Transfer learning curve', 'A'),
    (axes[1], 'spearman_rho',  'Spearman \u03c1', 'B   Transfer learning curve (Spearman)', 'B'),
]:
    ax.axhline(zs, color='#95a5a6', lw=1, linestyle=':', label=f'Zero-shot ({zs:.3f})')
    ax.axhline(full_r if metric == 'pearson_r' else 0.459,
               color='#27ae60', lw=1, linestyle='--',
               label=f'Full training ({full_r:.3f})')
    ax.plot(ft['fraction'], ft[metric], 'o-', color='#e74c3c',
            lw=1.8, ms=5, label='SAINTdb → fine-tune')
    ax.fill_between(ft['fraction'],
                    ft[metric] * 0.97, ft[metric] * 1.03,  # ±3% band as visual guide
                    alpha=0.15, color='#e74c3c')
    ax.set_xlabel('AAYL51 training fraction used for fine-tuning')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=6)
    ax.set_xlim(-0.02, 0.85)
    ax.grid(True, alpha=0.3)
    for _, row in ft.iterrows():
        ax.annotate(f'{row[metric]:.3f}', (row.fraction, row[metric]),
                    textcoords='offset points', xytext=(0, 5), fontsize=5.5, ha='center')

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig13_transfer_learning')
plt.show()


---
## 14. Top-K Recovery — Practical Screening Metric

What fraction of true top-K% binders does Ours recover in its predicted top-K%? Compared against random baseline.

In [ ]:
topk = pd.read_csv(os.path.join(ADV, 'topk_recovery.csv'))
print(topk.to_string(index=False))


In [ ]:
# Figure 14: Top-K recovery — fixed random baseline, proper panels
topk = pd.read_csv(os.path.join(ADV, 'topk_recovery_fixed.csv'))
print(topk.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

# Panel A: Grouped bar per k level
ax = axes[0]
k_labels = [f'Top-{int(r.k_pct*100)}\%' for _, r in topk.iterrows()]
x = np.arange(len(k_labels)); w = 0.22
methods = [
    ('random_recovery',    '#bdc3c7', 'Random (theoretical)'),
    ('zero_shot_recovery', '#3498db', 'Zero-shot (SAINTdb)'),
    ('in_domain_recovery', '#e74c3c', 'In-domain (full-chain)'),
    ('allcdr_recovery',    '#2ecc71', 'In-domain (All-CDR)'),
]
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
for (col, color, label), offset in zip(methods, offsets):
    if col in topk.columns:
        vals = topk[col].values
        bars = ax.bar(x + offset, vals, w, color=color, alpha=0.87, label=label)
        for bar, v in zip(bars, vals):
            if v > 0.01:
                ax.text(bar.get_x()+bar.get_width()/2, v+0.008,
                        f'{v:.2f}', ha='center', fontsize=5, rotation=90)

ax.set_xticks(x); ax.set_xticklabels(k_labels, fontsize=7)
ax.set_ylabel('Recovery fraction (recall)')
ax.set_title('Top-K recovery\n(fraction of true top-K binders retrieved)')
ax.legend(fontsize=6, ncol=2)
ax.set_ylim(0, 0.75)
ax.axhline(1.0, color='gray', lw=0.5, ls='--', alpha=0.3)
ax.text(0.02, 0.97, 'A', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top')

# Panel B: Enrichment factor = recovery / random
ax = axes[1]
for (col, color, label), offset in zip(methods[1:], offsets[1:]):
    if col in topk.columns:
        enrichment = topk[col].values / topk['random_recovery'].values
        ax.plot(topk['k_pct']*100, enrichment, 'o-', color=color,
                lw=1.8, ms=5, label=label)
        for k, e in zip(topk['k_pct']*100, enrichment):
            ax.text(k, e+0.05, f'{e:.1f}x', ha='center', fontsize=5.5)
ax.axhline(1.0, color='gray', lw=1, ls='--', label='Random (1×)')
ax.set_xlabel('K (%)')
ax.set_ylabel('Enrichment factor (recovery / random)')
ax.set_title('Screening enrichment over random\n(higher = better)')
ax.legend(fontsize=6)
ax.set_xticks([5,10,20,30])
ax.text(0.02, 0.97, 'B', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top')

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig14_topk_recovery')
plt.show()

---
## 15. In Silico Saturation Mutagenesis — AAYL51 CDR Positions

Using all 4320 measured CDR variants from the AAYL51 dataset. Shows which positions and substitutions cause the largest affinity changes.

In [ ]:
hm = pd.read_csv(os.path.join(ADV, 'cdr_heatmap_data.csv'))
cdr_perf = pd.read_csv(os.path.join(ADV, 'cdr_region_performance.csv'))
print("CDR region performance:")
print(cdr_perf[['cdr','n_variants','n_positions','pearson_r','spearman_rho']].to_string(index=False))
print(f"\nHeatmap data: {len(hm)} position×AA combinations")


In [ ]:
# Figure 15: CDR mutagenesis heatmap — fixed (all mutations per sequence)
hm      = pd.read_csv(os.path.join(ADV, 'cdr_heatmap_data_fixed.csv'))
cdr_pf  = pd.read_csv(os.path.join(ADV, 'cdr_region_performance_fixed.csv'))
wt_ref  = pd.read_csv(os.path.join(ADV, 'wt_reference.csv'))
wt_pkd  = float(wt_ref['wt_pkd'].values[0])
print("CDR region performance:"); print(cdr_pf.to_string(index=False))
print(f"\nHeatmap entries: {hm.groupby('cdr_region').size().to_dict()}")

AAs  = list('ACDEFGHIKLMNPQRSTVWY')
regs = ['CDR-H1', 'CDR-H2', 'CDR-H3']
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
fig.suptitle('In Silico Saturation Mutagenesis — AAYL51 CDR Variants\n'
             '(4320 real CDR variants; marginal effect per mutation position)',
             fontsize=9, fontweight='bold')

for col_idx, (region, letter) in enumerate(zip(regs, 'ABC')):
    sub = hm[hm['cdr_region'] == region]
    pos = sorted(sub['mut_pos'].unique())
    perf = cdr_pf[cdr_pf['cdr'] == region]
    n_var  = int(perf['n_variants'].values[0]) if len(perf) else 0
    r_val  = float(perf['pearson_r'].values[0]) if len(perf) else np.nan
    rho_val= float(perf['spearman_rho'].values[0]) if len(perf) else np.nan

    # ── Top row: measured ΔpKD ──────────────────────────────────────
    ax = axes[0, col_idx]
    mat = np.full((len(AAs), len(pos)), np.nan)
    for j, p in enumerate(pos):
        for i, aa in enumerate(AAs):
            row = sub[(sub['mut_pos'] == p) & (sub['mut_aa'] == aa)]
            if len(row):
                mat[i, j] = float(row['delta_pkd_true'].values[0])
    vmax = max(abs(np.nanmin(mat)), abs(np.nanmax(mat)), 0.1)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im = ax.imshow(mat, cmap='RdYlGn', norm=norm, aspect='auto')
    ax.set_yticks(range(len(AAs))); ax.set_yticklabels(AAs, fontsize=5.5)
    wt_labels = [f'{sub[sub.mut_pos==p]["wt_aa"].iloc[0]}{p}'
                 if len(sub[sub.mut_pos==p]) else f'?{p}' for p in pos]
    ax.set_xticks(range(len(pos))); ax.set_xticklabels(wt_labels, fontsize=5, rotation=90)
    ax.set_xlabel('Position (WT residue + index)')
    ax.set_ylabel('Substituted AA' if col_idx == 0 else '')
    ax.set_title(f'{letter}   {region} — Measured ΔpKD\n'
                 f'n={n_var} variants  r={r_val:.3f}  ρ={rho_val:.3f}', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.75, label=r'ΔpK$_d$', pad=0.02)
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', color='black',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

    # ── Bottom row: BALM predicted ΔpKD ─────────────────────────────
    ax2 = axes[1, col_idx]
    mat2 = np.full((len(AAs), len(pos)), np.nan)
    for j, p in enumerate(pos):
        for i, aa in enumerate(AAs):
            row = sub[(sub['mut_pos'] == p) & (sub['mut_aa'] == aa)]
            if len(row):
                mat2[i, j] = float(row['delta_pkd_pred'].values[0])
    vmax2 = max(abs(np.nanmin(mat2)), abs(np.nanmax(mat2)), 0.05)
    norm2 = TwoSlopeNorm(vmin=-vmax2, vcenter=0, vmax=vmax2)
    im2   = ax2.imshow(mat2, cmap='RdYlGn', norm=norm2, aspect='auto')
    ax2.set_yticks(range(len(AAs))); ax2.set_yticklabels(AAs, fontsize=5.5)
    ax2.set_xticks(range(len(pos))); ax2.set_xticklabels(wt_labels, fontsize=5, rotation=90)
    ax2.set_xlabel('Position (WT + index)')
    ax2.set_ylabel('Substituted AA' if col_idx == 0 else '')
    ax2.set_title(f'{chr(ord(letter)+3)}   {region} — BALM predicted ΔpKD', fontsize=8)
    plt.colorbar(im2, ax=ax2, shrink=0.75, label=r'Pred ΔpK$_d$', pad=0.02)
    ax2.text(0.02, 0.97, chr(ord(letter)+3), transform=ax2.transAxes,
             fontsize=10, fontweight='bold', va='top', color='black',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()
save_fig(fig, 'fig15_cdr_mutagenesis_heatmap')
plt.show()

---
## 16. Hybrid Pipeline: BALM + Boltz-2 Affinity Head

Training a small regression head on the combined feature vector of BALM predictions plus Boltz-2 structural metrics (ipTM, ipLDDT, ipDE, contact count, cross-chain PAE). Evaluated by leave-one-out CV on 150 top candidates from cascade screening.

In [ ]:
ah = pd.read_csv(os.path.join(ADV, 'affinity_head_results.csv'))
print("Affinity head results (LOO-CV on 150 cascade candidates):")
print(ah.to_string(index=False))


In [ ]:
# Figure 16: Hybrid affinity head + cascade funnel
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))

# Panel A: Performance comparison bar chart
ax = axes[0]
x  = np.arange(len(ah))
ah_colors = ['#3498db', '#e67e22', '#e74c3c', '#9b59b6'][:len(ah)]
bars_p = ax.bar(x - 0.2, ah['pearson'],  0.38, color=ah_colors, alpha=0.85, label='Pearson r')
bars_s = ax.bar(x + 0.2, ah['spearman'], 0.38, color=ah_colors, alpha=0.55, label='Spearman \u03c1')
ax.set_xticks(x)
ax.set_xticklabels(ah['model'], fontsize=6, rotation=20, ha='right')
ax.set_ylabel('Correlation')
ax.set_title('A   Affinity head performance\n(LOO-CV, n=150 cascade candidates)')
ax.legend(fontsize=6)
ax.axhline(0, color='k', lw=0.5)
for bar, val in zip(bars_p, ah['pearson']):
    ax.text(bar.get_x()+bar.get_width()/2, val+(0.005 if val>=0 else -0.015),
            f'{val:.3f}', ha='center', fontsize=5.5)

# Panel B: Cascade funnel diagram
cas = pd.read_csv(os.path.join(HERE, 'cascade_results/final_reranked.csv'))
ax = axes[1]
stages = ['All AAYL51\nvariants', 'BALM top-150\ncandidates', 'Boltz-2 validated\n(all 150)', 'Top-20\nby combined']
counts = [4320, 150, 150, 20]
true_pkd_means = [
    float(pd.read_csv(os.path.join(HERE, 'cascade_results/stage1_all_ranked.csv'))['pKD'].mean()),
    float(cas['pKD'].mean()),
    float(cas['pKD'].mean()),
    float(cas.head(20)['pKD'].mean()),
]
ax2 = ax.twinx()
x_f  = np.arange(len(stages))
bars = ax.bar(x_f, counts, color=['#95a5a6','#3498db','#f39c12','#e74c3c'], alpha=0.8, width=0.6)
ax2.plot(x_f, true_pkd_means, 'ko-', lw=1.5, ms=5, label='Mean true pK$_d$')
ax.set_xticks(x_f); ax.set_xticklabels(stages, fontsize=6)
ax.set_ylabel('Number of variants (log scale)')
ax.set_yscale('log')
ax2.set_ylabel('Mean true pK$_d$', color='k')
ax.set_title('B   Cascade screening funnel\n(Ours → Boltz-2 → re-rank)')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, n*1.2, str(n), ha='center', fontsize=6.5)
ax2.legend(loc='upper left', fontsize=6)

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig16_hybrid_affinity_head')
plt.show()


---
## 17. Compute Scalability

Inference time comparison: Ours (sequence-based) vs Boltz-2 (structure-based). Emphasizes BALM's practical advantage for large-scale library screening.

In [ ]:
timing = pd.read_csv(os.path.join(ADV, 'compute_timing.csv'))
print(timing[['method','n_variants','total_sec','per_variant_ms']].to_string(index=False))
balm_ms  = float(timing[timing.method.str.contains('BALM')]['per_variant_ms'].values[0])
boltz_ms = float(timing[timing.method.str.contains('Boltz')]['per_variant_ms'].values[0])
print(f"\nSpeedup: BALM is {boltz_ms/balm_ms:.0f}x faster per variant")
print(f"10,000 variants: BALM ~{10000*balm_ms/1000:.0f}s  |  Boltz-2 ~{10000*boltz_ms/3600:.0f}h")


In [ ]:
# Figure 17: Compute scalability
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

# Panel A: Per-variant time comparison (log scale)
ax = axes[0]
methods = ['Ours (Ours)\n(GPU batch)', 'ESMFold\n(estimated)', 'Boltz-2\n(1 recycle)', 'AlphaFold2\n(estimated)']
ms_times = [balm_ms, 500., boltz_ms, 120_000.]
bar_colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
bars = ax.bar(range(len(methods)), ms_times, color=bar_colors, alpha=0.85, width=0.65)
ax.set_yscale('log')
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, fontsize=6.5)
ax.set_ylabel('Inference time per variant (ms, log scale)')
ax.set_title('A   Per-variant inference time\n(lower = faster)')
for bar, val in zip(bars, ms_times):
    label = f'{val:.1f}ms' if val < 1000 else f'{val/1000:.1f}s'
    ax.text(bar.get_x()+bar.get_width()/2, val*1.3, label, ha='center', fontsize=6)

# Panel B: Throughput for library screening
ax = axes[1]
library_sizes = [100, 1_000, 10_000, 100_000, 1_000_000]
balm_hours  = [n * balm_ms / 3_600_000 for n in library_sizes]
boltz_hours = [n * boltz_ms / 3_600_000 for n in library_sizes]
ax.loglog(library_sizes, balm_hours,  'o-', color='#2ecc71', lw=1.8, ms=5, label=r'Ours (Ours (Ours (Ours-AbAg)))')
ax.loglog(library_sizes, boltz_hours, 's-', color='#e74c3c', lw=1.8, ms=5, label='Boltz-2')
ax.axhline(1, color='gray', lw=0.8, linestyle='--', alpha=0.5, label='1 hour')
ax.axhline(24, color='gray', lw=0.8, linestyle=':', alpha=0.5, label='24 hours')
ax.set_xlabel('Library size (number of variants)')
ax.set_ylabel('Compute time (hours, log scale)')
ax.set_title('B   Screening throughput\n(practical library sizes)')
ax.legend(fontsize=6.5)
ax.grid(True, which='both', alpha=0.2)

plt.tight_layout(w_pad=2)
save_fig(fig, 'fig17_compute_scalability')
plt.show()


---
## 18. Complete Results Summary

In [ ]:
# Full results table
rows = []
# CDR ablation
for _, r in cdr_abl.iterrows():
    rows.append({'Experiment': f'CDR ablation: {r.mode}',
                 'Pearson r': f'{r.indomain_r:.4f}', 'Spearman \u03c1': f'{r.indomain_rho:.4f}',
                 'Note': f'zero-shot r={r.zeroshot_r:.4f}'})

# Transfer learning
for _, r in tl.iterrows():
    if r.fraction in (0.0, 0.10, 0.30, 0.80):
        rows.append({'Experiment': f'Transfer {r.mode} frac={r.fraction:.2f}',
                     'Pearson r': f'{r.pearson_r:.4f}', 'Spearman \u03c1': f'{r.spearman_rho:.4f}',
                     'Note': f'n_finetune={r.n_train}'})

# Affinity head
for _, r in ah.iterrows():
    rows.append({'Experiment': f'Affinity head: {r.model}',
                 'Pearson r': f'{r.pearson:.4f}', 'Spearman \u03c1': f'{r.spearman:.4f}',
                 'Note': 'LOO-CV n=150'})

final_df = pd.DataFrame(rows)
print(final_df.to_string(index=False))


In [ ]:
print('\nAll publication figures:')
for f in sorted(os.listdir(FIG_OUT)):
    if f.endswith('.png'):
        size = os.path.getsize(os.path.join(FIG_OUT, f)) // 1024
        print(f'  {f:50s} {size:5d} KB')


---
## New Analyses — Gating, Baselines, Statistical Tests, All-CDR

In [ ]:
# Shared constants for new figures
PKD = r'pK$_d$'          # proper axis label
DELTA_PKD = r'ΔpK$_d$'
MODEL_NAME = 'Ours (Ours)'
ADV = os.path.join(HERE, 'advanced_results')
print("New analysis cells loaded. PKD label:", PKD)

---
## 18. Gating Mechanism Ablation — Learned vs Fixed Gate

In [ ]:
gate_eb = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/gate_perturbation_errorbars.csv'))
print(gate_eb.head(8).to_string(index=False))
gate_modes = ['learned','fixed_0.5','open_1.0','closed_0.0','random_uniform']
gate_labels = ['Ours\n(learned gate)', 'Fixed 0.5', 'Open 1.0\n(no gating)', 'Closed 0.0\n(blocked)', 'Random']
gate_colors = ['#e74c3c','#3498db','#2ecc71','#95a5a6','#f39c12']

In [ ]:
# Figure 18: Gating analysis with 95% CI
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.5))
fig.suptitle(f'{MODEL_NAME}: Learned Gating vs Fixed Gate Ablation\nSAaIntDB 10-fold CV',
             fontsize=9, fontweight='bold')

def get_gate(split, metric):
    vals, cis = [], []
    for gm in gate_modes:
        row = gate_eb[(gate_eb.split==split)&(gate_eb.gate_mode==gm)&(gate_eb.metric==metric)]
        vals.append(float(row['mean'].values[0]) if len(row) else np.nan)
        cis.append(float(row['ci95'].values[0]) if len(row) else np.nan)
    return np.array(vals), np.array(cis)

x = np.arange(len(gate_modes))
for (ax, split, letter) in [(axes[0,0],'Random','A'),(axes[0,1],'Cold','B'),
                              (axes[1,0],'Random','C'),(axes[1,1],'Cold','D')]:
    metric, ylabel = ('pearson' if letter in 'AB' else 'rmse',
                      'Pearson r' if letter in 'AB' else f'RMSE ({PKD})')
    vals, cis = get_gate(split, metric)
    bars = ax.bar(x, vals, color=gate_colors, alpha=0.85, width=0.65,
                  yerr=cis, error_kw=dict(lw=1,capsize=3,capthick=0.8))
    ax.set_xticks(x); ax.set_xticklabels(gate_labels, fontsize=6)
    ax.set_ylabel(ylabel); ax.set_title(f'{letter}  {split} split — {ylabel}', fontsize=8)
    # Highlight Ours bar
    bars[0].set_edgecolor('black'); bars[0].set_linewidth(1.5)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+bar.get_width()/2, v+(0.005 if metric=='pearson' else 0.01),
                    f'{v:.3f}', ha='center', fontsize=5.5)
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    # Add dotted line at learned gate
    learned_val = vals[0]
    if not np.isnan(learned_val):
        ax.axhline(learned_val, color='#e74c3c', lw=0.8, ls=':', alpha=0.6)

plt.tight_layout()
save_fig(fig, 'fig18_gating_ablation')
plt.show()

---
## 19. Model Architecture Comparison with 95% CI and Fisher Tests

In [ ]:
fold_eb  = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/fold_level_errorbars_all_models.csv'))
boot_ci  = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/bootstrap_ci_permutation_tests.csv'))
fisher_df= pd.read_csv(os.path.join(ADV, 'fisher_tests.csv'))
print(fold_eb[fold_eb.metric=='pearson'][['model','mean','std','ci95']].to_string(index=False))
print("\nFisher tests:"); print(fisher_df.to_string(index=False))

In [ ]:
# Figure 19: Architecture comparison with CI bars
ARCH_MAP = {
    'BALM random':        ('Ours (random)',     '#e74c3c'),
    'BALM cold':          ('Ours (cold)',        '#c0392b'),
    'Concat+MLP random':  ('Concat+MLP (rand)', '#3498db'),
    'Concat+MLP cold':    ('Concat+MLP (cold)', '#2980b9'),
    'Two-stream random':  ('Two-stream (rand)', '#2ecc71'),
    'Two-stream cold':    ('Two-stream (cold)', '#27ae60'),
    'Random antigen':     ('Random antigen',    '#95a5a6'),
}

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.5))
fig.suptitle(f'{MODEL_NAME} vs Baselines — SAaIntDB 10-fold CV\n'
             '(bars = mean ± 95% CI; * p<0.05 Fisher z-test vs Ours)',
             fontsize=9, fontweight='bold')

for ax, metric, ylabel, letter in [
    (axes[0], 'pearson',  'Pearson r',     'A'),
    (axes[1], 'spearman', 'Spearman ρ',    'B'),
    (axes[2], 'rmse',     f'RMSE ({PKD})', 'C'),
]:
    rows = fold_eb[fold_eb.metric == metric]
    ordered = [(k, v) for k, v in ARCH_MAP.items() if k in rows.model.values]
    xs = np.arange(len(ordered))
    for i, (model_key, (label, color)) in enumerate(ordered):
        row = rows[rows.model == model_key].iloc[0]
        ax.bar(i, row['mean'], color=color, alpha=0.85, width=0.65,
               yerr=row['ci95'], error_kw=dict(lw=1, capsize=3))
        ax.text(i, row['mean']+(0.005 if metric!='rmse' else 0.01),
                f'{row["mean"]:.3f}', ha='center', fontsize=5.5)
        # Fisher significance vs Ours
        if 'random' in model_key and 'BALM' not in model_key and 'Random' not in model_key:
            ft_row = fisher_df[fisher_df.comparison.str.contains(
                model_key.replace(' random','').replace('+',' ').split()[0], na=False)]
            if len(ft_row) and bool(ft_row['significant'].values[0]):
                ax.text(i, row['mean']+(row['ci95'] or 0)+0.02, '*',
                        ha='center', fontsize=9, color='k', fontweight='bold')

    ax.set_xticks(xs)
    ax.set_xticklabels([v[0] for _, v in ordered], fontsize=5.5, rotation=25, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(f'{letter}  {ylabel}', fontsize=8)
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig19_architecture_comparison_ci')
plt.show()

---
## 20. Statistical Analysis — Fisher z-Tests & Per-Antigen CI

In [ ]:
per_pdb  = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/per_pdb_metrics.csv'))
intra_ag = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/per_antigen_intra_target_metrics.csv'))
fisher_df= pd.read_csv(os.path.join(ADV, 'fisher_tests.csv'))
print(f"Per-PDB: {len(per_pdb)} groups")
print(f"Intra-antigen: {len(intra_ag)} groups")
print("Fisher tests:"); print(fisher_df.to_string(index=False))

In [ ]:
# Figure 20: Statistical significance analysis
from scipy import stats as sp_stats

fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.5))
fig.suptitle('Statistical Analysis — Fisher z-Tests & Subgroup Performance\nSAaIntDB',
             fontsize=9, fontweight='bold')

# Panel A: Fisher test results as lollipop
ax = axes[0, 0]
if len(fisher_df) > 0:
    ax.barh(range(len(fisher_df)), fisher_df['z_stat'].abs(),
            color=['#e74c3c' if s else '#95a5a6' for s in fisher_df['significant']],
            alpha=0.85, height=0.6)
    ax.set_yticks(range(len(fisher_df)))
    ax.set_yticklabels(fisher_df['comparison'], fontsize=6)
    ax.axvline(1.96, color='k', lw=1, ls='--', label='z=1.96 (p=0.05)')
    ax.set_xlabel('|Fisher z-statistic|')
    ax.set_title('A  Fisher z-tests (red = significant)', fontsize=8)
    ax.legend(fontsize=6)
    for i, (_, row) in enumerate(fisher_df.iterrows()):
        ax.text(row['z_stat'].real + 0.1, i,
                f"p={row['p_value']:.3f}{'*' if row['significant'] else ''}",
                va='center', fontsize=6)
ax.text(0.02, 0.97, 'A', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel B: Per-antigen Pearson r distribution with CI
ax = axes[0, 1]
rand_intra = intra_ag[(intra_ag.split=='Random') & intra_ag.pearson.notna()]
sorted_intra = rand_intra.sort_values('pearson', ascending=False).reset_index(drop=True)
# Bootstrap CI per antigen (approximate from beta distribution)
ns = sorted_intra['n'].values
rs = sorted_intra['pearson'].values
# Fisher CI: tanh(arctanh(r) ± 1.96/sqrt(n-3))
z_r = np.arctanh(np.clip(rs, -0.999, 0.999))
ci_half = 1.96 / np.sqrt(np.maximum(ns - 3, 1))
ci_low  = np.tanh(z_r - ci_half)
ci_high = np.tanh(z_r + ci_half)
x_pos = np.arange(len(sorted_intra))
ax.scatter(x_pos, rs, s=20, c='#e74c3c', zorder=3)
ax.vlines(x_pos, ci_low, ci_high, color='#e74c3c', alpha=0.5, lw=1)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('Antigen group (sorted by Pearson r)')
ax.set_ylabel('Pearson r ± 95% CI')
ax.set_title(f'B  Per-antigen performance (n≥3, random split)', fontsize=8)
ax.text(0.02, 0.97, 'B', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel C: Per-PDB n vs Pearson r scatter
ax = axes[1, 0]
pdb_rand = per_pdb[per_pdb.split=='Random'].dropna(subset=['pearson'])
if len(pdb_rand) > 0:
    ax.scatter(pdb_rand['n'], pdb_rand['pearson'], alpha=0.4, s=12, c='#3498db')
    # Bin by n and show mean ± CI
    for n_thresh in [3, 5, 10]:
        subset = pdb_rand[pdb_rand['n'] == n_thresh]['pearson'].dropna()
        if len(subset) >= 3:
            ax.errorbar(n_thresh, subset.mean(), yerr=subset.std()*1.96/np.sqrt(len(subset)),
                        fmt='ro', ms=5, capsize=3, lw=1.5)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_xlabel('Number of variants per PDB')
    ax.set_ylabel('Pearson r')
    ax.set_title('C  Per-PDB performance vs group size', fontsize=8)
ax.text(0.02, 0.97, 'C', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel D: Bootstrap CI for main metrics across models
ax = axes[1, 1]
boot_ci = pd.read_csv(os.path.join(HERE, 'figures_out_saaintdb/additional_experiments/bootstrap_ci_permutation_tests.csv'))
models_boot = ['BALM_random', 'ConcatMLP_random', 'TwoStream_random']
model_labels = ['Ours', 'Concat+MLP', 'Two-stream']
colors_boot  = ['#e74c3c', '#3498db', '#2ecc71']
x_b = np.arange(len(models_boot)); w_b = 0.28
for i, (m, lbl, c) in enumerate(zip(models_boot, model_labels, colors_boot)):
    row = boot_ci[(boot_ci.model == m) & (boot_ci.metric == 'pearson')]
    if len(row):
        v  = float(row['point'].values[0])
        lo = float(row['ci95_low'].values[0])
        hi = float(row['ci95_high'].values[0])
        ax.bar(i, v, color=c, alpha=0.85, width=0.55)
        ax.errorbar(i, v, yerr=[[v-lo],[hi-v]], fmt='none',
                    ecolor='k', capsize=4, lw=1.5)
        ax.text(i, hi+0.005, f'{v:.3f}', ha='center', fontsize=6.5, fontweight='bold')
        # p-value
        p_row = boot_ci[(boot_ci.model == m) & (boot_ci.metric == 'pearson')]
        if len(p_row):
            pval = float(p_row['permutation_pvalue'].values[0])
            ax.text(i, v/2, f'p<0.001', ha='center', fontsize=5.5, color='white')
ax.set_xticks(range(len(models_boot))); ax.set_xticklabels(model_labels, fontsize=7)
ax.set_ylabel('Pearson r'); ax.set_title('D  Bootstrap 95% CI (1000 resamples)', fontsize=8)
ax.text(0.02, 0.97, 'D', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

plt.tight_layout()
save_fig(fig, 'fig20_statistical_analysis')
plt.show()

---
## 21. All-CDR Zero-Shot & Few-Shot — AAYL51

In [ ]:
tl_comb = pd.read_csv(os.path.join(ADV, 'tl_combined_comparison.csv'))
fisher_df = pd.read_csv(os.path.join(ADV, 'fisher_tests.csv'))
print(tl_comb.to_string(index=False))

In [ ]:
# Figure 21+22: All-CDR vs Mean-pool (zero-shot + few-shot combined)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
fig.suptitle(f'All-CDR Pooling vs Mean-pool — AAYL51 Transfer Learning\n'
             f'(864 test samples, same 80/20 split)',
             fontsize=9, fontweight='bold')

mp_data  = tl_comb[tl_comb.embedding == 'Mean-pool'].sort_values('fraction')
cdr_data = tl_comb[tl_comb.embedding == 'All-CDR'].sort_values('fraction')

# Panel A: Learning curves
ax = axes[0]
ax.plot(mp_data.fraction, mp_data.pearson_r, 'o-', color='#95a5a6',
        lw=1.8, ms=5, label='Mean-pool')
ax.plot(cdr_data.fraction, cdr_data.pearson_r, 's-', color='#e74c3c',
        lw=1.8, ms=5, label='All-CDR (H1+H2+H3)')
ax.axhline(0.4228, color='#e74c3c', lw=0.8, ls=':', alpha=0.5, label='In-domain max (0.423)')
ax.set_xlabel('Training fraction of AAYL51')
ax.set_ylabel('Pearson r'); ax.set_title(f'A  Transfer learning curve', fontsize=8)
ax.legend(fontsize=6); ax.grid(True, alpha=0.3)
ax.text(0.02, 0.97, 'A', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel B: Bar comparison at key fractions
ax = axes[1]
fracs_show = [0.0, 0.10, 0.30, 0.80]
x_b = np.arange(len(fracs_show)); w_b = 0.35
for offset, emb, color, label in [(-w_b/2,'Mean-pool','#95a5a6','Mean-pool'),
                                    (w_b/2, 'All-CDR', '#e74c3c', 'All-CDR')]:
    sub = tl_comb[tl_comb.embedding==emb].set_index('fraction')
    vals = [float(sub.loc[f,'pearson_r']) if f in sub.index else 0 for f in fracs_show]
    bars = ax.bar(x_b+offset, vals, w_b, color=color, alpha=0.85, label=label)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.3f}',
                ha='center', fontsize=5.5, rotation=90)
ax.set_xticks(x_b); ax.set_xticklabels([f'{int(f*100)}%' for f in fracs_show])
ax.set_xlabel('Training fraction'); ax.set_ylabel('Pearson r')
ax.set_title('B  Mean-pool vs All-CDR at key fractions', fontsize=8)
ax.legend(fontsize=6.5)
ax.text(0.02, 0.97, 'B', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel C: Δ(All-CDR − Mean-pool)
ax = axes[2]
mp_idx  = mp_data.set_index('fraction')['pearson_r']
cdr_idx = cdr_data.set_index('fraction')['pearson_r']
common_f = sorted(set(mp_idx.index) & set(cdr_idx.index))
deltas   = [float(cdr_idx.loc[f] - mp_idx.loc[f]) for f in common_f]
clrs     = ['#e74c3c' if d > 0 else '#3498db' for d in deltas]
ax.bar(range(len(common_f)), deltas, color=clrs, alpha=0.85, width=0.65)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(range(len(common_f)))
ax.set_xticklabels([f'{int(f*100)}%' for f in common_f])
ax.set_xlabel('Training fraction'); ax.set_ylabel('Δ Pearson r (All-CDR − Mean-pool)')
ax.set_title('C  CDR-aware gain', fontsize=8)
ax.text(0.02, 0.97, 'C', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig21_allcdr_transfer_learning')
plt.show()

---
## 22. All-CDR vs Mean-Pool: CDR Mutagenesis Performance

In [ ]:
# Figure 22: CDR region performance — Mean-pool vs All-CDR in-domain
cdr_perf = pd.read_csv(os.path.join(ADV, 'cdr_region_performance_fixed.csv'))
print(cdr_perf[['cdr','n_variants','n_positions','pearson_r','spearman_rho']].to_string(index=False))

In [ ]:
# Figure 22: Per-CDR region performance comparison
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
fig.suptitle(f'CDR Region Performance — In-Domain Model on AAYL51\n'
             f'(real mutagenesis data, 4320 CDR variants)',
             fontsize=9, fontweight='bold')

cdr_perf = pd.read_csv(os.path.join(ADV, 'cdr_region_performance_fixed.csv'))
hm_fixed = pd.read_csv(os.path.join(ADV, 'cdr_heatmap_data_fixed.csv'))
wt_ref   = pd.read_csv(os.path.join(ADV, 'wt_reference.csv'))
wt_pkd   = float(wt_ref['wt_pkd'].values[0])

for ax, (metric, ylabel, letter) in zip(axes, [
    ('pearson_r', 'Pearson r', 'A'),
    ('spearman_rho', 'Spearman ρ', 'B'),
    ('rmse', f'RMSE ({PKD})', 'C'),
]):
    if metric not in cdr_perf.columns: continue
    regions  = cdr_perf['cdr'].tolist()
    vals     = cdr_perf[metric].values
    n_vars   = cdr_perf['n_variants'].values
    bar_clrs = ['#e74c3c', '#3498db', '#2ecc71'][:len(regions)]
    bars = ax.bar(range(len(regions)), vals, color=bar_clrs, alpha=0.87, width=0.6)
    for i, (bar, v, n) in enumerate(zip(bars, vals, n_vars)):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}\n(n={n})',
                ha='center', fontsize=6, va='bottom')
    ax.set_xticks(range(len(regions))); ax.set_xticklabels(regions, fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(f'{letter}  Per-CDR region: {ylabel}', fontsize=8)
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_ylim(0, max(vals)*1.3)

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig22_cdr_region_performance')
plt.show()

---
## 23. SAaIntDB All-CDR: Detailed 10-Fold Analysis with CI

In [ ]:
# Figure 23: SAINTdb All-CDR per-fold + scatter
rand_folds = pd.read_csv(os.path.join(HERE, 'results_saaintdb_allcdr/random/cv_summary.csv'))
cold_folds = pd.read_csv(os.path.join(HERE, 'results_saaintdb_allcdr/cold/cv_summary.csv'))
allcdr_cmp = pd.read_csv(os.path.join(HERE, 'results_saaintdb_allcdr/saaintdb_allcdr_comparison.csv'))

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
fig.suptitle(f'{MODEL_NAME}: All-CDR H1+H2+H3 on SAaIntDB\n'
             f'ANARCI IMGT numbering — 10-fold CV',
             fontsize=9, fontweight='bold')

# Panel A: Per-fold Pearson r (random split) with CI
ax = axes[0]
folds_x  = np.arange(1, len(rand_folds)+1)
baseline_r = 0.8424  # mean-pool baseline
ax.bar(folds_x, rand_folds['pearson'], color='#e74c3c', alpha=0.8, width=0.7, label='All-CDR')
ax.axhline(baseline_r, color='#95a5a6', lw=1.5, ls='--', label=f'Mean-pool ({baseline_r:.3f})')
ax.axhline(rand_folds['pearson'].mean(), color='#c0392b', lw=1.2, ls=':',
           label=f'All-CDR mean ({rand_folds["pearson"].mean():.3f})')
ax.set_xticks(folds_x); ax.set_xlabel('Fold'); ax.set_ylabel('Pearson r')
ax.set_title('A  Per-fold (random split)', fontsize=8)
ax.legend(fontsize=6); ax.set_ylim(0.7, 1.0)
ax.text(0.02, 0.97, 'A', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel B: Summary comparison with CI
ax = axes[1]
methods  = ['Mean-pool\n(ESM-2)', 'All-CDR\n(H1+H2+H3)']
rand_vals= [0.8424, float(allcdr_cmp[allcdr_cmp.split=='random']['pearson'].iloc[-1])]
rand_stds= [0.0212, float(rand_folds['pearson'].std())]
cold_vals= [0.6000, float(allcdr_cmp[allcdr_cmp.split=='cold']['pearson'].iloc[-1])]
cold_stds= [0.0720, float(cold_folds['pearson'].std())]
x_m = np.arange(len(methods)); w_m = 0.35
b1 = ax.bar(x_m-w_m/2, rand_vals, w_m, yerr=[1.96*s for s in rand_stds],
            color=['#95a5a6','#e74c3c'], alpha=0.87, label='Random',
            error_kw=dict(lw=1, capsize=3))
b2 = ax.bar(x_m+w_m/2, cold_vals, w_m, yerr=[1.96*s for s in cold_stds],
            color=['#bdc3c7','#c0392b'], alpha=0.87, label='Cold',
            error_kw=dict(lw=1, capsize=3), hatch='//')
for bar, v in zip(list(b1)+list(b2), rand_vals+cold_vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.3f}',
            ha='center', fontsize=6, fontweight='bold')
ax.set_xticks(x_m); ax.set_xticklabels(methods); ax.set_ylabel('Pearson r')
ax.set_title('B  Random vs Cold split ± 95% CI', fontsize=8)
ax.legend(fontsize=6.5)
ax.text(0.02, 0.97, 'B', transform=ax.transAxes,
        fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
# Δ annotations
for i, (rv, cv) in enumerate(zip(rand_vals, cold_vals)):
    delta_r = rv - rand_vals[0] if i > 0 else 0
    ax.annotate(f'Δ={delta_r:+.3f}', xy=(i, max(rv,cv)+0.04),
                ha='center', fontsize=7,
                color='#e74c3c' if delta_r > 0 else '#3498db',
                fontweight='bold' if abs(delta_r) > 0.01 else 'normal')

# Panel C: Predicted vs true scatter for best fold
ax = axes[2]
best_fold = int(rand_folds.loc[rand_folds.pearson.idxmax(), 'fold'])
best_pred = pd.read_csv(os.path.join(HERE, f'results_saaintdb_allcdr/random/fold_{best_fold:02d}',
                                      '../all_preds.csv') if os.path.isfile(
    os.path.join(HERE, f'results_saaintdb_allcdr/random/all_preds.csv'))
    else os.path.join(HERE, 'results_saaintdb_allcdr/random/all_preds.csv'))
fold_pred = best_pred[best_pred.fold == best_fold] if 'fold' in best_pred.columns else best_pred
if 'predicted_affinity' in fold_pred.columns and 'binding_affinity' in fold_pred.columns:
    t = fold_pred['binding_affinity'].values; p = fold_pred['predicted_affinity'].values
    ax.scatter(t, p, c='#e74c3c', alpha=0.35, s=6, linewidths=0, rasterized=True)
    lims = [min(t.min(),p.min())-0.3, max(t.max(),p.max())+0.3]
    ax.plot(lims,lims,'k--',lw=0.8,alpha=0.5)
    r_b, _ = pearsonr(t,p)
    ax.set_xlabel(f'Measured {PKD}'); ax.set_ylabel(f'Predicted {PKD}')
    ax.set_title(f'C  Best fold scatter (fold {best_fold}, r={r_b:.3f})', fontsize=8)
    ax.text(0.02,0.97,'C',transform=ax.transAxes, fontsize=10, fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

plt.tight_layout(w_pad=1.5)
save_fig(fig, 'fig23_saaintdb_allcdr_detail')
plt.show()

---
## 24. Comprehensive Summary: All-CDR vs Mean-Pool Across All Tasks

In [ ]:
# Figure 24: Comprehensive comparison across all datasets/tasks
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
fig.suptitle(f'{MODEL_NAME}: Mean-pool ESM-2 vs All-CDR (H1+H2+H3) Pooling\n'
             f'Comprehensive comparison across tasks',
             fontsize=10, fontweight='bold')

tasks = {
    'SAaIntDB\n(random)':  {'mp': 0.8424, 'cdr': float(allcdr_cmp[allcdr_cmp.split=="random"]['pearson'].iloc[-1]),
                             'mp_std': 0.0212, 'cdr_std': rand_folds['pearson'].std()},
    'SAaIntDB\n(cold)':    {'mp': 0.6000, 'cdr': float(allcdr_cmp[allcdr_cmp.split=="cold"]['pearson'].iloc[-1]),
                             'mp_std': 0.0720, 'cdr_std': cold_folds['pearson'].std()},
    'AAYL51\n(in-domain)': {'mp': float(cdr_abl[cdr_abl['mode']=="Full-chain"]['indomain_r']),
                             'cdr': float(cdr_abl[cdr_abl['mode']=="All-CDR (H1+H2+H3)"]['indomain_r']),
                             'mp_std': 0.02, 'cdr_std': 0.02},
    'AAYL51\n(zero-shot)': {'mp': float(cdr_abl[cdr_abl['mode']=="Full-chain"]['zeroshot_r']),
                             'cdr': float(cdr_abl[cdr_abl['mode']=="All-CDR (H1+H2+H3)"]['zeroshot_r']),
                             'mp_std': 0.03, 'cdr_std': 0.03},
}

# Top row: bar comparison per task
ax = axes[0, 0]
x_t = np.arange(len(tasks)); w_t = 0.35
b_mp  = ax.bar(x_t-w_t/2, [v['mp']  for v in tasks.values()], w_t, color='#95a5a6',
               alpha=0.87, label='Mean-pool', yerr=[v['mp_std']*1.96 for v in tasks.values()],
               error_kw=dict(lw=1, capsize=3))
b_cdr = ax.bar(x_t+w_t/2, [v['cdr'] for v in tasks.values()], w_t, color='#e74c3c',
               alpha=0.87, label='All-CDR', yerr=[v['cdr_std']*1.96 for v in tasks.values()],
               error_kw=dict(lw=1, capsize=3))
ax.set_xticks(x_t); ax.set_xticklabels(list(tasks.keys()), fontsize=7)
ax.set_ylabel('Pearson r'); ax.set_title('A  Pearson r across tasks (±95% CI)', fontsize=8)
ax.legend(fontsize=7); ax.set_ylim(0, 1.05)
ax.text(0.02, 0.97, 'A', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Δ gains
ax2 = axes[0, 1]
deltas = [v['cdr']-v['mp'] for v in tasks.values()]
clrs_d = ['#e74c3c' if d>0 else '#3498db' for d in deltas]
ax2.barh(range(len(tasks)), deltas, color=clrs_d, alpha=0.85, height=0.6)
ax2.set_yticks(range(len(tasks))); ax2.set_yticklabels(list(tasks.keys()), fontsize=7)
ax2.axvline(0, color='k', lw=0.6)
ax2.set_xlabel('Δ Pearson r (All-CDR − Mean-pool)')
ax2.set_title('B  CDR-aware gain per task', fontsize=8)
for i, d in enumerate(deltas):
    ax2.text(d+(0.002 if d>=0 else -0.002), i, f'{d:+.3f}',
             va='center', ha='left' if d>=0 else 'right', fontsize=7, fontweight='bold')
ax2.text(0.02, 0.97, 'B', transform=ax2.transAxes, fontsize=10, fontweight='bold', va='top',
         bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel C: Transfer learning comparison (mean-pool vs All-CDR)
ax3 = axes[0, 2]
ax3.plot(mp_data.fraction, mp_data.pearson_r, 'o-', color='#95a5a6', lw=1.8, ms=5, label='Mean-pool')
ax3.plot(cdr_data.fraction, cdr_data.pearson_r, 's-', color='#e74c3c', lw=1.8, ms=5, label='All-CDR')
ax3.set_xlabel('AAYL51 training fraction'); ax3.set_ylabel('Pearson r')
ax3.set_title('C  Transfer learning: AAYL51 fine-tuning', fontsize=8)
ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)
ax3.text(0.02, 0.97, 'C', transform=ax3.transAxes, fontsize=10, fontweight='bold', va='top',
         bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel D-F: CDR ablation + top-K + SAINTdb fold scatter
for ax_d, (ax, src_ax) in enumerate([(axes[1,0], None), (axes[1,1], None), (axes[1,2], None)]):
    pass

# Panel D: CDR region pearson r
ax = axes[1, 0]
cdr_perf = pd.read_csv(os.path.join(ADV, 'cdr_region_performance_fixed.csv'))
bar_c = ['#e74c3c','#3498db','#2ecc71'][:len(cdr_perf)]
ax.bar(range(len(cdr_perf)), cdr_perf['pearson_r'], color=bar_c, alpha=0.87, width=0.6)
for i, (_, row) in enumerate(cdr_perf.iterrows()):
    ax.text(i, row['pearson_r']+0.01, f'{row["pearson_r"]:.3f}',
            ha='center', fontsize=7, fontweight='bold')
ax.set_xticks(range(len(cdr_perf)))
ax.set_xticklabels(cdr_perf['cdr'].tolist(), fontsize=8)
ax.set_ylabel('Pearson r'); ax.set_title('D  Per-CDR region performance (AAYL51)', fontsize=8)
ax.text(0.02, 0.97, 'D', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel E: Top-K recovery All-CDR vs Mean-pool
ax = axes[1, 1]
topk_f = pd.read_csv(os.path.join(ADV, 'topk_recovery_fixed.csv'))
x_k = np.arange(len(topk_f)); w_k = 0.22
for col, color, label, off in [
    ('random_recovery',   '#bdc3c7', 'Random',          -0.33),
    ('in_domain_recovery','#95a5a6', 'Mean-pool (full)',  0.0),
    ('allcdr_recovery',   '#e74c3c', 'All-CDR in-domain',0.33),
]:
    if col in topk_f.columns:
        ax.bar(x_k+off, topk_f[col], w_k, color=color, alpha=0.87, label=label)
ax.set_xticks(x_k); ax.set_xticklabels([f'Top-{int(r*100)}%' for r in topk_f['k_pct']])
ax.set_ylabel('Recovery fraction'); ax.set_title('E  Top-K recovery comparison', fontsize=8)
ax.legend(fontsize=5.5, ncol=1); ax.set_ylim(0, 0.7)
ax.text(0.02, 0.97, 'E', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Panel F: Compute speedup
ax = axes[1, 2]
timing = pd.read_csv(os.path.join(ADV, 'compute_timing.csv'))
balm_ms  = float(timing[timing.method.str.contains('BALM')]['per_variant_ms'].values[0])
boltz_ms = float(timing[timing.method.str.contains('Boltz')]['per_variant_ms'].values[0])
methods_t = ['Ours\n(GPU)', 'ESMFold\n(est.)', 'Boltz-2\n(1 recycle)', 'AF2\n(est.)']
times_ms  = [balm_ms, 500., boltz_ms, 120000.]
bar_ct    = ['#e74c3c','#3498db','#f39c12','#9b59b6']
ax.bar(range(len(methods_t)), times_ms, color=bar_ct, alpha=0.85, width=0.6)
ax.set_yscale('log'); ax.set_xticks(range(len(methods_t)))
ax.set_xticklabels(methods_t, fontsize=7)
ax.set_ylabel('ms/variant (log scale)')
ax.set_title('F  Compute: Ours vs alternatives', fontsize=8)
for i, v in enumerate(times_ms):
    ax.text(i, v*1.4, f'{v:.1f}ms' if v<1000 else f'{v/1000:.0f}s',
            ha='center', fontsize=6)
ax.text(0.02, 0.97, 'F', transform=ax.transAxes, fontsize=10, fontweight='bold', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

plt.tight_layout()
save_fig(fig, 'fig24_comprehensive_summary')
plt.show()

---
# Part IV — In-Silico Saturation Mutagenesis with the FINAL All-CDR Model

Regenerated using the **All-CDR (H1+H2+H3) model weights** (`advanced_results/allcdr_indomain_model.pt`) and All-CDR embeddings, replacing the earlier full-chain version. Marginal ΔpK$_d$ per CDR position × substituted amino acid, on the 4320 real AAYL51 CDR variants.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
ADV=os.path.join(HERE,'advanced_results')
hm=pd.read_csv(os.path.join(ADV,'cdr_heatmap_data_allcdr.csv'))
perf=pd.read_csv(os.path.join(ADV,'cdr_region_performance_allcdr.csv'))
print('All-CDR per-region performance:'); print(perf.to_string(index=False))

In [ ]:
AAs=list('ACDEFGHIKLMNPQRSTVWY'); regs=['CDR-H1','CDR-H2','CDR-H3']
fig,axes=plt.subplots(1,3,figsize=(11,5))
for ax,reg,letter in zip(axes,regs,'ABC'):
    sub=hm[hm.cdr_region==reg]; pos=sorted(sub['mut_pos'].unique())
    mat=np.full((len(AAs),len(pos)),np.nan)
    for j,p in enumerate(pos):
        for i,aa in enumerate(AAs):
            r=sub[(sub.mut_pos==p)&(sub.mut_aa==aa)]
            if len(r): mat[i,j]=float(r['delta_pkd_true'].values[0])
    vmax=max(abs(np.nanmin(mat)),abs(np.nanmax(mat)),0.1); norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
    im=ax.imshow(mat,cmap='RdYlGn',norm=norm,aspect='auto')
    ax.set_yticks(range(len(AAs))); ax.set_yticklabels(AAs,fontsize=5.5)
    wtl=[f"{sub[sub.mut_pos==p]['wt_aa'].iloc[0]}{p}" if len(sub[sub.mut_pos==p]) else f'?{p}' for p in pos]
    ax.set_xticks(range(len(pos))); ax.set_xticklabels(wtl,fontsize=5,rotation=90)
    pr=perf[perf.cdr==reg]; rv=float(pr['pearson_r'].values[0]) if len(pr) else np.nan
    nv=int(pr['n_variants'].values[0]) if len(pr) else 0
    ax.set_xlabel('Position (WT+idx)'); ax.set_ylabel('Substituted AA' if letter=='A' else '')
    ax.set_title(f'{letter}  {reg} — All-CDR (n={nv}, r={rv:.3f})',fontsize=8)
    plt.colorbar(im,ax=ax,shrink=0.7,label=r'ΔpK$_d$')
fig.suptitle('In-silico saturation mutagenesis — FINAL All-CDR model (AAYL51)',fontsize=9,fontweight='bold')
plt.tight_layout(); save_fig(fig,'fig25_allcdr_mutagenesis'); plt.show()

In [ ]:
# All-CDR vs full-chain mutagenesis correlation comparison
old=pd.read_csv(os.path.join(ADV,'cdr_region_performance_fixed.csv'))
fig,ax=plt.subplots(figsize=(4.6,3.0)); x=np.arange(3); w=0.38
o=[float(old[old.cdr==r]['pearson_r'].values[0]) for r in regs]
a=[float(perf[perf.cdr==r]['pearson_r'].values[0]) for r in regs]
ax.bar(x-w/2,o,w,color='#95a5a6',alpha=0.85,label='Full-chain')
ax.bar(x+w/2,a,w,color='#e74c3c',alpha=0.85,label='All-CDR (final)')
ax.set_xticks(x); ax.set_xticklabels(regs); ax.set_ylabel('Pearson r (within CDR)')
ax.set_title('Mutagenesis: All-CDR vs full-chain',fontsize=8.5); ax.legend(fontsize=6.5)
for i in range(3):
    ax.text(i-w/2,o[i]+0.01,f'{o[i]:.2f}',ha='center',fontsize=6)
    ax.text(i+w/2,a[i]+0.01,f'{a[i]:.2f}',ha='center',fontsize=6)
plt.tight_layout(); save_fig(fig,'fig26_mutagenesis_allcdr_vs_fullchain'); plt.show()